# Claim Rx Monthly Features — Cohort-Based

Refactored from `01_claim_rx_monthly` (NBA 3.0) to support **configurable cohort windows with offset**.

* **Offset = 0** reproduces the original calendar-month behavior
* **Offset = 7 / 14 / 21** slides all windows forward by that many days
* **Lookback windows** (1, 3, 6, 12 months, as 3.0) are date-range-based, not row-based

Source notebook: `/Users/wenjie.chen@bayer.com/kerendia_nba_3_0_develop/NBA 3.0 Ops - Pipeline/03_Feature_Engineering/03_HCP_Level_Features/01_claim_rx_monthly`

## Part 1 — Reverse Engineering of `01_claim_rx_monthly`

### Pipeline Overview

```
df_rx (daily claims)    pat_new_and_total_treatments (monthly)
       │                              │
       ├─ date_trunc(‘MM’, SVC_DT) ─────────┤
       │   → SVC_DT_M (month grain)    │
       │                              │
       ├─ 8 Feature Groups (pivot+count per NPI per month)
       │   2.1  Treatment Category    ─┐
       │   2.2  Brand Level           │
       │   2.3  Rejection             │
       │   2.4  Duration (Tx×DaysSupply)│
       │   2.5  Refill (Tx×RefillFlag)│
       │   2.6  Payer Plan (Tx×Payment)│
       │   2.7  Patient Attributes    │
       │   2.8  Patient Treatment     │
       │                              │
       ├─ Combine (left-join on NPI,month) + SGLT2_WO_INVOKANA derived cols
       │
       ├─ step2_fill_missing → cross-join NPI×months, left-join features
       │
       ├─ step3_lag → Window(rowsBetween) for SUM in last [1,3,6,12] months
       │               + RUNNING totals
       │
       ├─ PCT features (feature / total_claims, per window)
       │
       ├─ AVG features (avg_age, avg_treatment_per_patient) with step3_lag_avg
       │
       ├─ Difference features (_IN_LAST_3_6_MONTH = last_6 − last_3)
       │   Ratio features (prior_3 / last_3)
       │
       ├─ Filter SVC_DT_M >= 2022-01-01
       │
       └─ Write → claim_rx_monthly (Delta)
              → Join NPI→BH_ID → claim_rx_monthly_provider (Delta)
```

**Output grain**: One row per (NPI_NUMBER, SVC_DT_M) — ~3.3M rows  
**Output location**: `/Volumes/.../notebook_output/claim_rx_monthly`

### Data Sources

| Source | Path | Grain | Key Columns Used |
|--------|------|-------|------------------|
| Rx claims | `.../query_output/Rx_01_new` | 1 row per claim | NPI_NUMBER, SVC_DT, TREATMENT_CATEGORY, Category, REJECTION_FLAG, DAYS_SUPPLY, REFILL_FLAG, METHOD_OF_PAYMENT, PATIENT_ID, PATIENT_BIRTH_YEAR, PATIENT_GENDER |
| Patient treatments | `.../query_output/pat_new_and_total_treatments` | 1 row per (PATIENT_ID, month) | PATIENT_ID, SVC_DT_M, TOTAL_TREATMENTS, TOTAL_NEW_TREATMENTS |
| NPI mapping | `.../kerendia_nba_3_volume/bh_id_npi_mapping` | 1 row per NPI | NPI_NUMBER, BH_ID |

### Feature Groups (Step 2 in original)

All feature groups follow the same pattern: **groupBy(NPI, month) → pivot(dimension) → count claims → fill zeros**.

Column naming convention: `TOTAL_RX_CLAIMS_{PIVOT_VALUE}` (or `TOTAL_REJECTED_RX_CLAIMS_` for rejections).

| # | Group | Pivot Column(s) | Filter | Prefix | Cell |
|---|-------|----------------|--------|--------|------|
| 2.1 | Treatment Category | TREATMENT_CATEGORY | none | `TOTAL_RX_CLAIMS_` | 12 |
| 2.2 | Brand Level | Category | Category IN (‘INVOKANA 100MG’, ‘INVOKANA 300MG’, ‘FARXIGA’, ‘JARDIANCE’, ‘SPIRONOLACTONE-MRA’, ‘Diuretic + MRA’, ‘DPP-IV’, ‘GLP-1’) | `TOTAL_RX_CLAIMS_` | 15 |
| 2.3 | Rejection | TREATMENT_CATEGORY + Category (same filters as 2.1+2.2) | REJECTION_FLAG = 1 | `TOTAL_REJECTED_RX_CLAIMS_` | 18 |
| 2.4 | Duration | TREATMENT_CATEGORY×DAYS_SUPPLY + Category×DAYS_SUPPLY | none (DAYS_SUPPLY bucketed to 30/60/90/Not/Unknown) | `TOTAL_RX_CLAIMS_` | 20–28 |
| 2.5 | Refill | TREATMENT_CATEGORY×REFILL_FLAG + Category×REFILL_FLAG + REFILL_FLAG alone | none | `TOTAL_RX_CLAIMS_` | 30 |
| 2.6 | Payer Plan | TREATMENT_CATEGORY×METHOD_OF_PAYMENT + Category×METHOD_OF_PAYMENT + METHOD_OF_PAYMENT alone | none (null → Unknown) | `TOTAL_RX_CLAIMS_` | 32 |
| 2.7 | Patient Attributes | (Tx×AGE_BIN, Cat×AGE_BIN, AGE_BIN, Tx×GENDER, Cat×GENDER, GENDER) | none (AGE bucketed to 10-yr bins, Gender null → Unknown) | `TOTAL_RX_CLAIMS_` | 35 |
| 2.8 | Patient Treatment | TOTAL_ACTIVE_TREATMENTS bucket, TOTAL_NEW_TREATMENTS bucket | none (joined with pat_new_and_total_treatments) | `TOTAL_RX_CLAIMS_PATIENT_WITH_` | 37–38 |

### Derived Features (after combine)

| Feature Pattern | Logic | Cell |
|----------------|-------|------|
| `SGLT2_WO_INVOKANA_300MG` variants | SGLT2 − INVOKANA_300MG (for counts, duration, rejection, age, gender) | 40 |
| `_IN_LAST_{1,3,6,12}_MONTH` | Rolling SUM via `Window.rowsBetween(-i+1, 0)` on monthly rows | 44–45 |
| `RUNNING_` prefix | Unbounded preceding SUM | 45 |
| `PCT_` prefix | feature / TOTAL_RX_CLAIMS (per window) | 48–52 |
| `AVG_PATIENT_AGE_IN_LAST_{i}_MONTH` | Rolling AVG via `Window.rowsBetween(-i, 0)` | 55–56 |
| `AVG_TREATMENT_PER_PATIENT_IN_LAST_{i}_MONTH` | Rolling AVG | 55–56 |
| `TOTAL_PATIENT_COUNT_IN_LAST_{i}_MONTH` | Rolling AVG (oddly — averages monthly counts) | 55–56 |
| `_IN_LAST_3_6_MONTH` | last_6 − last_3 ("prior 3 months") | 60 |
| `RATIO_..._IN_PRIOR_3_LAST_3_MONTH` | prior_3 / last_3 (safe div) | 61 |

### Where Monthly Logic Is Hardcoded (⭐ Refactor Targets)

| Location | Code | What It Does | Impact |
|----------|------|-------------|--------|
| **Cell 7** | `date_trunc('MM', SVC_DT)` | Truncates daily claims to first-of-month | ⭐ Must keep daily `SVC_DT` for offset > 0; month truncation only valid for offset = 0 |
| **step2_fill_missing** (helper) | `sequence(min_date, max_date, interval 1 month)` | Creates monthly grid for cross-join | ⭐ Replaced entirely — no grid needed when computing per date-range window |
| **step3_lag** (helper) | `Window.rowsBetween(-i+1, 0)` | Rolling SUM over last i months (row = month) | ⭐ Replaced by date-range filter + aggregate per window |
| **step3_lag_avg** (helper) | `Window.rowsBetween(-i, 0)` | Rolling AVG over last i+1 months | ⭐ Same replacement |
| **Cell 62** | `filter(to_date('SVC_DT_M') >= '2022-01-01')` | Hardcoded start date filter | ⭐ Replaced by cohort config |
| **Cell 37** | `join(pat_new_and_total_treatments, on=['PATIENT_ID', 'SVC_DT_M'])` | Joins monthly patient treatments | ✅ Resolved — `assign_cohort_month()` aligns the join key with the patient journey's offset-based SVC_DT_M |
| **Cell 35** | `F.year(col('SVC_DT_M')) - PATIENT_BIRTH_YEAR` | Age from month year | Minor — can use `SVC_DT` year instead |

### ⚠️ Flags for Wenjie

1. ~~**`pat_new_and_total_treatments`** month alignment~~ **RESOLVED**: `assign_cohort_month()` (added to helper functions) maps claim dates to offset-aligned cohort months, matching the patient journey pipeline's `SVC_DT_M`. Applied in cells 18 (windowed patient treatment), 19 (RUNNING patient treatment), and 20 (AVG monthly aggregation). No daily-grain version needed.
2. **`00_write_input_files`**: The `Rx_01_new` source table already has daily `SVC_DT` — no change needed. But if the write notebook creates any monthly aggregates, those may need updating.
3. **`RUNNING_` features**: In the cohort model, unbounded-preceding running totals don’t have a clear interpretation (running from when?). Consider dropping or redefining as running total within a fixed horizon (e.g., 24 months).
4. **`step2_fill_missing` and `step3_lag/step3_lag_avg`** in `00_step23_function` are no longer needed for this notebook. Other notebooks (02–06) still use them until they are also refactored.

## Part 2 — Cohort-Based Design

### How the Offset Works

```
Calendar:     Jan     Feb     Mar     Apr     May
              |-------|-------|-------|-------|

Cohort = Mar’26, Offset = 0:
  Cutoff = 31 Mar
  Last 1m:  1 Mar ─────── 31 Mar
  Last 3m:  1 Jan ─────────────── 31 Mar

Cohort = Mar’26, Offset = 7:
  Cutoff = 7 Apr
  Last 1m:  7 Mar ─────── 7 Apr
  Last 3m:  7 Jan ─────────────── 7 Apr
```

The entire lookback slides forward by the offset. At offset = 0, windows align with calendar months.

### Output Grain

**Old**: One row per (NPI_NUMBER, SVC_DT_M) — one per calendar month  
**New**: One row per (NPI_NUMBER, COHORT_ID) — one per cohort run

For training, generate multiple cohorts (C1 = Mar’26, C2 = Feb’26, …). For scoring, generate the current cohort only.

### Feature Naming Convention (preserved)

Base window features keep the same `_IN_LAST_{N}_MONTH` suffix to maintain downstream compatibility. The cohort config records which dates each window actually covers.

### What Changed vs. Original

| Aspect | Original (3.0) | Cohort-Based (3.1) |
|--------|---------------|--------------------|
| Time grain | Calendar month | Date range from cutoff |
| Aggregation | `date_trunc` → monthly pivot → row-window | Date-range filter → direct pivot per window |
| Window definition | `rowsBetween(-i+1, 0)` on monthly rows | `SVC_DT BETWEEN cutoff-Nm AND cutoff` |
| Fill missing | Cross-join NPI×months | NPI universe from max window; outer-join per window |
| RUNNING features | Unbounded preceding sum | ⚠️ Dropped — redefine if needed |
| Parameters | None (hardcoded) | COHORT_MONTH, OFFSET_DAYS, LOOKBACK_WINDOWS |

## Part 3 — Implementation

### 0. Configuration

**Cell order note**: The `%run` cells MUST execute before the cohort parameters cell.
`00_connection` may call `restartPython()`, which clears all prior variables.

In [0]:
%run "/Workspace/Users/wenjie.chen@bayer.com/kerendia_nba_3_0_develop/NBA 3.0 Ops - Pipeline/00_Helper_Notebooks/00_connection"

In [0]:
%run "../00_cohort_config"

In [0]:
# All config (OFFSET_DAYS, COHORT_MONTHS, LOOKBACK_WINDOWS, NPI_COL, SVC_COL, VALIDATION_MODE,
# get_cohort_months, get_cohort_bounds, build_feature_group, assign_cohort_month, apply_npi_filter, ...)
# comes from 00_cohort_config.

_cohort_setting = (COHORT_MONTHS if COHORT_MONTHS is not None
                   else f"all {COHORT_ORIGIN[:7]} .. {COHORT_LAST[:7]} (complete data only)")
print(f"Offset: {OFFSET_DAYS} | Cohorts: {_cohort_setting} | Validation: {VALIDATION_MODE}")

In [0]:
# ---------- Load raw Rx claims (daily level) ----------
# Paths, brands and patient tables come from 00_cohort_config (HCP shared constants)
_QUERY_OUTPUT    = HCP_QUERY_OUTPUT
_NOTEBOOK_OUTPUT = HCP_NOTEBOOK_OUTPUT
df_rx = spark.read.format("delta").load(_QUERY_OUTPUT + "Rx_01_new")

# Same cleanup as original
df_rx = df_rx.withColumn(
    'TREATMENT_CATEGORY', F.regexp_replace('TREATMENT_CATEGORY', r'[\s]*$', '')
)

# Last service date in the full Rx data (before the NPI filter) -> which cohorts are complete
DATA_MAX_DT = df_rx.agg(F.max(SVC_COL)).first()[0]

# Validation mode: restrict to the 10 test HCPs (no-op when VALIDATION_MODE = False).
# Same filter as NB02, applied before any feature so every step sees the same NPIs.
df_rx = apply_npi_filter(df_rx, NPI_COL)

# pat_new_and_total_treatments is monthly, keyed on offset-aligned SVC_DT_M from the patient journey.
# assign_cohort_month() is used for the join so months align at any offset (patient treatment cells).
# Validation mode reads the 3.1 validation output of the patient journey notebooks; it has to be
# rebuilt at the current OFFSET_DAY before offset > 0 results are final.
# Production still reads the 3.0 table until a 3.1 production output exists.
_PAT_TX_TABLE = get_patient_table("new_and_total_treatments")
pat_new_and_total_treatments = spark.read.format("delta").load(_QUERY_OUTPUT + _PAT_TX_TABLE)
print(f"Patient treatments: {_PAT_TX_TABLE}")

BRAND_FILTER = HCP_BRAND_FILTER   # original brand filter, built from HCP_BRANDS (00_cohort_config)

# ---------- Derived columns ----------
# Same definitions as the original per-group preps (duration, payer, patient attributes),
# added once so windowed, RUNNING_ and AVG features all use the same columns.
def add_rx_derived_cols(df):
    return (
        df
        .withColumn("DAYS_SUPPLY_BIN",
            F.when(F.col("DAYS_SUPPLY").rlike(r"^\s*30(\.0+)?\s*$"), "30_DAYS_SUPPLY")
            .when(F.col("DAYS_SUPPLY").rlike(r"^\s*60(\.0+)?\s*$"), "60_DAYS_SUPPLY")
            .when(F.col("DAYS_SUPPLY").rlike(r"^\s*90(\.0+)?\s*$"), "90_DAYS_SUPPLY")
            .when(F.col("DAYS_SUPPLY").isNull(), "DAYS_SUPPLY_Unknown")
            .otherwise("Not_30_60_90_Days_Supply"))
        .withColumn("METHOD_OF_PAYMENT",
            F.when(F.col("METHOD_OF_PAYMENT").isNull(), "METHOD_OF_PAYMENT_Unknown")
            .otherwise(F.col("METHOD_OF_PAYMENT")))
        .withColumn("AGE",
            F.when((F.col("PATIENT_BIRTH_YEAR") == 0) | (F.col("PATIENT_BIRTH_YEAR").isNull()), None)
            .otherwise(F.year(F.col("SVC_DT")) - F.col("PATIENT_BIRTH_YEAR")))
        .withColumn("AGE_ROUNDED", (F.floor(F.col("AGE") / 10) * 10))
        .withColumn("AGE_BIN",
            F.when((F.col("PATIENT_BIRTH_YEAR") == 0) | (F.col("PATIENT_BIRTH_YEAR").isNull()), "AGE_GROUP_IGNORE")
            .otherwise(
                F.when(F.col("AGE_ROUNDED") >= 80, "AGE_GROUP_ABOVE_80")
                .otherwise(
                    F.concat(F.lit("AGE_GROUP_"),
                             F.col("AGE_ROUNDED").cast("int").cast("string"),
                             F.lit("_"),
                             (F.col("AGE_ROUNDED") + 10).cast("int").cast("string")))))
        .withColumn("PATIENT_GENDER",
            F.when((F.col("PATIENT_GENDER") == "null") | (F.col("PATIENT_GENDER").isNull()), "PATIENT_GENDER_Unknown")
            .otherwise(F.col("PATIENT_GENDER")))
    )

df_rx = add_rx_derived_cols(df_rx)

# ---------- Data start date (team rule: no data before DATA_START_DATE) ----------
# Applies to every feature, RUNNING_ included, so RUNNING_ patient-treatment features only use months
# the 3.1 patient journey covers (it starts at the same date).
# df_rx_all keeps the full history: used only for the fixed pivot values (same columns as 3.0) and
# for the offset-0 parity check against the original (which used the full history).
df_rx_all = df_rx
df_rx = df_rx.filter(F.col(SVC_COL) >= F.to_date(F.lit(DATA_START_DATE)))
print(f"Claims from {DATA_START_DATE} (DATA_START_DATE); full history kept as df_rx_all for parity only")

# ---------- Cohorts to process ----------
# Every cohort (COHORT_ORIGIN..COHORT_LAST, or the COHORT_MONTHS list) whose cutoff is on or
# before the common last service date of all HCP inputs, so all its windows have complete data
# and every HCP feature table has the same COHORT_IDs.
COHORT_LIST   = get_hcp_cohort_months()   # common end date of all HCP inputs (00_cohort_config); DATA_MAX_DT kept for parity
COHORT_BOUNDS = {m: get_cohort_bounds(m) for m in COHORT_LIST}
COHORT_IDS    = [COHORT_BOUNDS[m]["cohort_id"] for m in COHORT_LIST]
PREVIEW_COHORT = COHORT_MONTH if COHORT_MONTH in COHORT_LIST else COHORT_LIST[0]
cb_preview     = COHORT_BOUNDS[PREVIEW_COHORT]
print(f"Rx data through {DATA_MAX_DT} -> {len(COHORT_LIST)} complete cohorts: "
      f"{COHORT_LIST[0]} .. {COHORT_LIST[-1]} | preview cohort: {PREVIEW_COHORT}")

# ---------- Pre-filter to the widest AVG range over all cohorts ----------
# AVG windows are 1 month wider when AVG_EXTRA_MONTH=True. Each cohort is then sliced to its own
# range with cohort_slice() (= the original single-cohort df_rx_filtered).
_max_window = builtins.max(LOOKBACK_WINDOWS)
_max_start  = builtins.min(COHORT_BOUNDS[m]["avg_window_bounds"][_max_window]["start"] for m in COHORT_LIST)
_end        = builtins.max(COHORT_BOUNDS[m]["avg_window_bounds"][_max_window]["end"] for m in COHORT_LIST)

df_rx_filtered = df_rx.filter(
    (F.col("SVC_DT") > F.lit(_max_start)) &
    (F.col("SVC_DT") <= F.lit(_end))
)
# NOTE: .cache() removed — not supported on serverless compute

print(f"Filtered claims to ({_max_start}, {_end}]")
print(f"Row count: {df_rx_filtered.count():,}")

In [0]:
# ---- Fixed pivot values: every cohort gets the same feature columns ----
# Collected once from the full Rx history (like the original monthly pivot over all months), with one
# pass over the distinct value combinations. Values with no claims in a window become 0.
# Combined pivots replicate concat_ws("_", ...) as used in build_feature_group (nulls skipped).
import pandas as pd

_combo_cols = ["TREATMENT_CATEGORY", "Category", "DAYS_SUPPLY_BIN", "REFILL_FLAG",
               "METHOD_OF_PAYMENT", "AGE_BIN", "PATIENT_GENDER"]
_combos = (
    # full history -> same columns as 3.0 even if a value only occurs before DATA_START_DATE
    df_rx_all.select(*[F.col(c) for c in _combo_cols],
                 F.expr(BRAND_FILTER).alias("_IS_BRAND"),
                 F.expr("REJECTION_FLAG = 1").alias("_IS_REJ"))
    .distinct()
    .toPandas()
)
_combos.columns = _combo_cols + ["_IS_BRAND", "_IS_REJ"]
_BRAND = _combos["_IS_BRAND"].fillna(False).astype(bool)
_REJ   = _combos["_IS_REJ"].fillna(False).astype(bool)

def _present(v):
    return v is not None and not pd.isna(v)

def _pivot_values(cols, mask=None):
    d = _combos if mask is None else _combos[mask]
    if len(cols) == 1:
        return sorted({v for v in d[cols[0]] if _present(v)})
    return sorted({"_".join(str(v) for v in row if _present(v)) for row in d[cols].itertuples(index=False)})

PIVOT_VALUES = {
    "category":        _pivot_values(["TREATMENT_CATEGORY"]),
    "brand":           _pivot_values(["Category"], _BRAND),
    "rej_category":    _pivot_values(["TREATMENT_CATEGORY"], _REJ),
    "rej_brand":       _pivot_values(["Category"], _REJ & _BRAND),
    "dur_category":    _pivot_values(["TREATMENT_CATEGORY", "DAYS_SUPPLY_BIN"]),
    "dur_brand":       _pivot_values(["Category", "DAYS_SUPPLY_BIN"], _BRAND),
    "refill_brand":    _pivot_values(["Category", "REFILL_FLAG"], _BRAND),
    "refill_alone":    _pivot_values(["REFILL_FLAG"]),
    "payer_category":  _pivot_values(["TREATMENT_CATEGORY", "METHOD_OF_PAYMENT"]),
    "payer_brand":     _pivot_values(["Category", "METHOD_OF_PAYMENT"], _BRAND),
    "payer_alone":     _pivot_values(["METHOD_OF_PAYMENT"]),
    "age_category":    _pivot_values(["TREATMENT_CATEGORY", "AGE_BIN"]),
    "age_brand":       _pivot_values(["Category", "AGE_BIN"], _BRAND),
    "age_alone":       _pivot_values(["AGE_BIN"]),
    "gender_category": _pivot_values(["TREATMENT_CATEGORY", "PATIENT_GENDER"]),
    "gender_brand":    _pivot_values(["Category", "PATIENT_GENDER"], _BRAND),
    "gender_alone":    _pivot_values(["PATIENT_GENDER"]),
    # Patient-treatment buckets are fixed by their definition (patient treatment cell)
    "active_tx": ["NO_ACTIVE_TREATMENT", "1_ACTIVE_TREATMENT", "2_OR_3_ACTIVE_TREATMENTS", "3_MORE_ACTIVE_TREATMENTS"],
    "new_tx":    ["NO_NEW_TREATMENT", "1_NEW_TREATMENT", "2_OR_3_NEW_TREATMENTS", "3_MORE_NEW_TREATMENTS"],
}

print(f"{len(_combos):,} distinct value combinations")
display(pd.DataFrame([{"group": k, "n_values": len(v), "values": ", ".join(map(str, v))}
                      for k, v in PIVOT_VALUES.items()]))

### 1. Build Feature Groups

Each group uses `build_feature_group` with date-range filters instead of monthly pivot + row-window.

In [0]:
# Each step below is a function of one cohort's bounds (cb = get_cohort_bounds(month)).
# The preview at the end of each cell runs it on cb_preview only; section 3 runs every cohort.

def cohort_slice(df, cb):
    """Claims in this cohort's widest AVG window = the original single-cohort df_rx_filtered."""
    _b = cb["avg_window_bounds"][builtins.max(LOOKBACK_WINDOWS)]
    return filter_to_window(df, SVC_COL, _b["start"], _b["end"])


def build_base_groups(df_c, cb):
    wb = cb["window_bounds"]
    # ---- NPI universe (windowed claims only — widened with df_running NPIs when combining) ----
    # Filter null NPIs — source data has ~16.7M null-NPI claims that are useless
    npi_universe = df_c.filter(F.col(NPI_COL).isNotNull()).select(NPI_COL).distinct()

    # ---- Total Rx Claims per window (denominator for PCT) ----
    df_total = build_total_col(df_c, NPI_COL, SVC_COL, wb)

    # ---- 2.1 Treatment Category ----
    df_category = build_feature_group(
        df_c, NPI_COL, SVC_COL,
        pivot_col="TREATMENT_CATEGORY",
        prefix="TOTAL_RX_CLAIMS_",
        window_bounds=wb,
        pivot_values=PIVOT_VALUES["category"]
    )

    # ---- 2.2 Brand Level ----
    df_brand = build_feature_group(
        df_c, NPI_COL, SVC_COL,
        pivot_col="Category",
        prefix="TOTAL_RX_CLAIMS_",
        window_bounds=wb,
        extra_filter=BRAND_FILTER,
        pivot_values=PIVOT_VALUES["brand"]
    )
    return npi_universe, df_total, df_category, df_brand


# ---- Preview on one cohort ----
df_rx_c = cohort_slice(df_rx_filtered, cb_preview)
npi_universe, df_total, df_category, df_brand = build_base_groups(df_rx_c, cb_preview)
print(f"Preview {cb_preview['cohort_id']} | NPI universe: {npi_universe.count():,} providers")
print(f"Treatment Category cols: {len(df_category.columns) - 1}")
print(f"Brand cols: {len(df_brand.columns) - 1}")

In [0]:
def build_rejection_duration(df_c, cb):
    wb = cb["window_bounds"]
    # ---- 2.3 Rejection (Treatment Category) ----
    df_rej_category = build_feature_group(
        df_c, NPI_COL, SVC_COL,
        pivot_col="TREATMENT_CATEGORY",
        prefix="TOTAL_REJECTED_RX_CLAIMS_",
        window_bounds=wb,
        extra_filter="REJECTION_FLAG = 1",
        pivot_values=PIVOT_VALUES["rej_category"]
    )

    # ---- 2.3 Rejection (Brand) ----
    df_rej_brand = build_feature_group(
        df_c, NPI_COL, SVC_COL,
        pivot_col="Category",
        prefix="TOTAL_REJECTED_RX_CLAIMS_",
        window_bounds=wb,
        extra_filter=f"REJECTION_FLAG = 1 AND {BRAND_FILTER}",
        pivot_values=PIVOT_VALUES["rej_brand"]
    )

    df_rejection = df_rej_category.join(df_rej_brand, on=NPI_COL, how="outer").na.fill(0)

    # ---- 2.4 Duration — pivot on Tx×Supply and Cat×Supply ----
    # DAYS_SUPPLY_BIN is added once in the load cell (add_rx_derived_cols), same bins as before.
    df_dur_category = build_feature_group(
        df_c, NPI_COL, SVC_COL,
        pivot_col=None,
        prefix="TOTAL_RX_CLAIMS_",
        window_bounds=wb,
        combine_cols=["TREATMENT_CATEGORY", "DAYS_SUPPLY_BIN"],
        pivot_values=PIVOT_VALUES["dur_category"]
    )

    df_dur_brand = build_feature_group(
        df_c, NPI_COL, SVC_COL,
        pivot_col=None,
        prefix="TOTAL_RX_CLAIMS_",
        window_bounds=wb,
        extra_filter=BRAND_FILTER,
        combine_cols=["Category", "DAYS_SUPPLY_BIN"],
        pivot_values=PIVOT_VALUES["dur_brand"]
    )

    df_duration = df_dur_category.join(df_dur_brand, on=NPI_COL, how="outer").na.fill(0)
    return df_rejection, df_duration


# ---- Preview on one cohort ----
df_rejection, df_duration = build_rejection_duration(df_rx_c, cb_preview)
print(f"Rejection cols: {len(df_rejection.columns) - 1}")
print(f"Duration cols: {len(df_duration.columns) - 1}")

In [0]:
def build_refill_payer(df_c, cb):
    wb = cb["window_bounds"]
    # ---- 2.5 Refill ----
    # Old notebook had brand × refill + refill alone only (no treatment × refill).
    df_refill_brand = build_feature_group(
        df_c, NPI_COL, SVC_COL,
        pivot_col=None,
        prefix="TOTAL_RX_CLAIMS_",
        window_bounds=wb,
        extra_filter=BRAND_FILTER,
        combine_cols=["Category", "REFILL_FLAG"],
        pivot_values=PIVOT_VALUES["refill_brand"]
    )
    df_refill_alone = build_feature_group(
        df_c, NPI_COL, SVC_COL,
        pivot_col="REFILL_FLAG",
        prefix="TOTAL_RX_CLAIMS_",
        window_bounds=wb,
        pivot_values=PIVOT_VALUES["refill_alone"]
    )
    df_refill = (
        df_refill_alone
        .join(df_refill_brand, on=NPI_COL, how="outer")
        .na.fill(0)
    )

    # ---- 2.6 Payer Plan ----
    # Null payment -> METHOD_OF_PAYMENT_Unknown is applied once in add_rx_derived_cols.
    df_payer_cat = build_feature_group(
        df_c, NPI_COL, SVC_COL,
        pivot_col=None,
        prefix="TOTAL_RX_CLAIMS_",
        window_bounds=wb,
        combine_cols=["TREATMENT_CATEGORY", "METHOD_OF_PAYMENT"],
        pivot_values=PIVOT_VALUES["payer_category"]
    )
    df_payer_brand = build_feature_group(
        df_c, NPI_COL, SVC_COL,
        pivot_col=None,
        prefix="TOTAL_RX_CLAIMS_",
        window_bounds=wb,
        extra_filter=BRAND_FILTER,
        combine_cols=["Category", "METHOD_OF_PAYMENT"],
        pivot_values=PIVOT_VALUES["payer_brand"]
    )
    df_payer_alone = build_feature_group(
        df_c, NPI_COL, SVC_COL,
        pivot_col="METHOD_OF_PAYMENT",
        prefix="TOTAL_RX_CLAIMS_",
        window_bounds=wb,
        pivot_values=PIVOT_VALUES["payer_alone"]
    )
    df_payer = (
        df_payer_alone
        .join(df_payer_cat, on=NPI_COL, how="outer")
        .join(df_payer_brand, on=NPI_COL, how="outer")
        .na.fill(0)
    )
    return df_refill, df_payer


# ---- Preview on one cohort ----
df_refill, df_payer = build_refill_payer(df_rx_c, cb_preview)
print(f"Refill cols: {len(df_refill.columns) - 1}")
print(f"Payer cols: {len(df_payer.columns) - 1}")

In [0]:
# ---- 2.7 Patient Attributes ----
# AGE / AGE_BIN and PATIENT_GENDER cleaning are applied once in add_rx_derived_cols
# (same logic as the original), so df_c already has them.
def build_patient_attributes(df_c, cb):
    wb = cb["window_bounds"]
    # Age pivots: Tx×Age, Cat×Age, Age alone
    df_pat_age_cat = build_feature_group(
        df_c, NPI_COL, SVC_COL, pivot_col=None,
        prefix="TOTAL_RX_CLAIMS_", window_bounds=wb,
        combine_cols=["TREATMENT_CATEGORY", "AGE_BIN"],
        pivot_values=PIVOT_VALUES["age_category"]
    )
    df_pat_age_brand = build_feature_group(
        df_c, NPI_COL, SVC_COL, pivot_col=None,
        prefix="TOTAL_RX_CLAIMS_", window_bounds=wb,
        extra_filter=BRAND_FILTER,
        combine_cols=["Category", "AGE_BIN"],
        pivot_values=PIVOT_VALUES["age_brand"]
    )
    df_pat_age_alone = build_feature_group(
        df_c, NPI_COL, SVC_COL, pivot_col="AGE_BIN",
        prefix="TOTAL_RX_CLAIMS_", window_bounds=wb,
        pivot_values=PIVOT_VALUES["age_alone"]
    )

    # Gender pivots: Tx×Gender, Cat×Gender, Gender alone
    df_pat_gen_cat = build_feature_group(
        df_c, NPI_COL, SVC_COL, pivot_col=None,
        prefix="TOTAL_RX_CLAIMS_", window_bounds=wb,
        combine_cols=["TREATMENT_CATEGORY", "PATIENT_GENDER"],
        pivot_values=PIVOT_VALUES["gender_category"]
    )
    df_pat_gen_brand = build_feature_group(
        df_c, NPI_COL, SVC_COL, pivot_col=None,
        prefix="TOTAL_RX_CLAIMS_", window_bounds=wb,
        extra_filter=BRAND_FILTER,
        combine_cols=["Category", "PATIENT_GENDER"],
        pivot_values=PIVOT_VALUES["gender_brand"]
    )
    df_pat_gen_alone = build_feature_group(
        df_c, NPI_COL, SVC_COL, pivot_col="PATIENT_GENDER",
        prefix="TOTAL_RX_CLAIMS_", window_bounds=wb,
        pivot_values=PIVOT_VALUES["gender_alone"]
    )

    return (
        df_pat_age_alone
        .join(df_pat_age_cat, on=NPI_COL, how="outer")
        .join(df_pat_age_brand, on=NPI_COL, how="outer")
        .join(df_pat_gen_alone, on=NPI_COL, how="outer")
        .join(df_pat_gen_cat, on=NPI_COL, how="outer")
        .join(df_pat_gen_brand, on=NPI_COL, how="outer")
        .na.fill(0)
    )


# ---- Preview on one cohort ----
df_patient = build_patient_attributes(df_rx_c, cb_preview)
print(f"Patient attribute cols: {len(df_patient.columns) - 1}")

In [0]:
# ---- 2.8 Patient Treatment (active/new treatment buckets) ----
# pat_new_and_total_treatments is monthly, keyed on offset-aligned SVC_DT_M from NB06.
# We use assign_cohort_month() to match the patient journey's month assignment,
# so claims near boundaries (e.g. Jan 5 with offset=7) map to the correct cohort month.

def build_patient_treatment_features(df_rx, pat_treatments, npi_col, svc_dt_col, window_bounds, brand_filter):
    """Build patient treatment bucket features across all windows."""
    window_dfs = []
    
    for w in sorted(window_bounds.keys()):
        bounds = window_bounds[w]
        df_w = filter_to_window(df_rx, svc_dt_col, bounds["start"], bounds["end"])
        
        # Join with patient treatments on offset-aligned cohort month
        df_w = df_w.withColumn("_SVC_DT_M", assign_cohort_month(F.col(svc_dt_col)))
        df_w = df_w.join(
            pat_treatments,
            (df_w["PATIENT_ID"] == pat_treatments["PATIENT_ID"]) &
            (df_w["_SVC_DT_M"] == pat_treatments["SVC_DT_M"]),
            how="left"
        ).drop(pat_treatments["PATIENT_ID"]).drop(pat_treatments["SVC_DT_M"])
        
        # Bucket treatment counts (same as original)
        df_w = df_w.withColumn("TOTAL_ACTIVE_TREATMENTS",
            F.when((F.col("TOTAL_TREATMENTS") < 1) | (F.col("TOTAL_TREATMENTS").isNull()), "NO_ACTIVE_TREATMENT")
            .otherwise(F.when(F.col("TOTAL_TREATMENTS") == 1, "1_ACTIVE_TREATMENT")
            .otherwise(F.when(F.col("TOTAL_TREATMENTS") < 3, "2_OR_3_ACTIVE_TREATMENTS")
            .otherwise("3_MORE_ACTIVE_TREATMENTS")))
        ).withColumn("TOTAL_NEW_TREATMENTS",
            F.when((F.col("TOTAL_NEW_TREATMENTS") < 1) | (F.col("TOTAL_NEW_TREATMENTS").isNull()), "NO_NEW_TREATMENT")
            .otherwise(F.when(F.col("TOTAL_NEW_TREATMENTS") == 1, "1_NEW_TREATMENT")
            .otherwise(F.when(F.col("TOTAL_NEW_TREATMENTS") < 3, "2_OR_3_NEW_TREATMENTS")
            .otherwise("3_MORE_NEW_TREATMENTS")))
        )
        
        suffix = f"_IN_LAST_{w}_MONTH"
        
        # Active treatments pivot
        df_active = pivoted_counts(df_w, npi_col, "TOTAL_ACTIVE_TREATMENTS",
                                    "TOTAL_RX_CLAIMS_PATIENT_WITH_", suffix,
                                    values=PIVOT_VALUES["active_tx"])
        # New treatments pivot
        df_new = pivoted_counts(df_w, npi_col, "TOTAL_NEW_TREATMENTS",
                                 "TOTAL_RX_CLAIMS_PATIENT_WITH_", suffix,
                                 values=PIVOT_VALUES["new_tx"])
        
        df_combined = df_active.join(df_new, on=npi_col, how="outer").na.fill(0)
        window_dfs.append(df_combined)
    
    # Join all windows
    result = window_dfs[0]
    for df_next in window_dfs[1:]:
        result = result.join(df_next, on=npi_col, how="outer")
    return result.na.fill(0)

# ---- Preview on one cohort ----
df_patient_treatment = build_patient_treatment_features(
    df_rx_c, pat_new_and_total_treatments,
    NPI_COL, SVC_COL, cb_preview["window_bounds"], BRAND_FILTER
)

print(f"Patient treatment cols: {len(df_patient_treatment.columns) - 1}")

In [0]:
# ---- RUNNING_ Features from full Rx history through the cohort cutoff ----
# Reproduces the old notebook's unbounded-preceding RUNNING_ columns.
# Uses the full Rx_01_new history (df_rx), not the windowed slice, and excludes null NPIs
# so the widened doctor universe is based only on real providers.
# DAYS_SUPPLY_BIN, payer Unknown, AGE_BIN and gender cleaning come from add_rx_derived_cols
# (same definitions as the windowed features).

def build_running_total(df_in, npi_col):
    return (
        df_in.groupBy(npi_col)
        .agg(F.count(F.lit(1)).alias("RUNNING_TOTAL_RX_CLAIMS"))
    )


def build_running_feature_group(df_in, npi_col, pivot_col, prefix, extra_filter=None,
                                combine_cols=None, pivot_values=None):
    """Build one RUNNING_ feature group over full history through the cohort cutoff."""
    df_work = df_in
    if extra_filter:
        df_work = df_work.filter(extra_filter)
    actual_pivot = pivot_col
    if combine_cols:
        df_work = df_work.withColumn(
            "_pivot_combined",
            F.concat_ws("_", *[F.col(c) for c in combine_cols])
        )
        actual_pivot = "_pivot_combined"
    return pivoted_counts(df_work, npi_col, actual_pivot, f"RUNNING_{prefix}", "", values=pivot_values)


# Same groups (and column order) as the original RUNNING_ cell:
# (PIVOT_VALUES key, pivot_col, prefix, extra_filter, combine_cols)
RUNNING_GROUPS = [
    ("category",        "TREATMENT_CATEGORY", "TOTAL_RX_CLAIMS_",          None,                                   None),
    ("brand",           "Category",           "TOTAL_RX_CLAIMS_",          BRAND_FILTER,                           None),
    ("rej_category",    "TREATMENT_CATEGORY", "TOTAL_REJECTED_RX_CLAIMS_", "REJECTION_FLAG = 1",                   None),
    ("rej_brand",       "Category",           "TOTAL_REJECTED_RX_CLAIMS_", f"REJECTION_FLAG = 1 AND {BRAND_FILTER}", None),
    ("dur_category",    None,                 "TOTAL_RX_CLAIMS_",          None,         ["TREATMENT_CATEGORY", "DAYS_SUPPLY_BIN"]),
    ("dur_brand",       None,                 "TOTAL_RX_CLAIMS_",          BRAND_FILTER, ["Category", "DAYS_SUPPLY_BIN"]),
    # Refill: old notebook had brand × refill + refill alone only (no treatment × refill)
    ("refill_alone",    "REFILL_FLAG",        "TOTAL_RX_CLAIMS_",          None,         None),
    ("refill_brand",    None,                 "TOTAL_RX_CLAIMS_",          BRAND_FILTER, ["Category", "REFILL_FLAG"]),
    ("payer_alone",     "METHOD_OF_PAYMENT",  "TOTAL_RX_CLAIMS_",          None,         None),
    ("payer_category",  None,                 "TOTAL_RX_CLAIMS_",          None,         ["TREATMENT_CATEGORY", "METHOD_OF_PAYMENT"]),
    ("payer_brand",     None,                 "TOTAL_RX_CLAIMS_",          BRAND_FILTER, ["Category", "METHOD_OF_PAYMENT"]),
    ("age_alone",       "AGE_BIN",            "TOTAL_RX_CLAIMS_",          None,         None),
    ("age_category",    None,                 "TOTAL_RX_CLAIMS_",          None,         ["TREATMENT_CATEGORY", "AGE_BIN"]),
    ("age_brand",       None,                 "TOTAL_RX_CLAIMS_",          BRAND_FILTER, ["Category", "AGE_BIN"]),
    ("gender_alone",    "PATIENT_GENDER",     "TOTAL_RX_CLAIMS_",          None,         None),
    ("gender_category", None,                 "TOTAL_RX_CLAIMS_",          None,         ["TREATMENT_CATEGORY", "PATIENT_GENDER"]),
    ("gender_brand",    None,                 "TOTAL_RX_CLAIMS_",          BRAND_FILTER, ["Category", "PATIENT_GENDER"]),
]


def build_running_patient_treatment(df_rx_running):
    """Patient treatment buckets: join claims to the monthly patient-treatment table on cohort month."""
    # Rename join keys on the right side to avoid AMBIGUOUS_REFERENCE on Spark Connect
    # (.drop(df["col"]) is unreliable when the plan contains prior broadcast joins).
    _pat_tx = (
        pat_new_and_total_treatments
        .withColumnRenamed("PATIENT_ID", "_pat_PATIENT_ID")
        .withColumnRenamed("SVC_DT_M", "_pat_SVC_DT_M")
    )
    df_tx_joined = (
        df_rx_running
        .withColumn("_SVC_DT_M", assign_cohort_month(F.col(SVC_COL)))
        .join(
            _pat_tx,
            (F.col("PATIENT_ID") == F.col("_pat_PATIENT_ID")) &
            (F.col("_SVC_DT_M") == F.col("_pat_SVC_DT_M")),
            how="left"
        )
        .drop("_pat_PATIENT_ID", "_pat_SVC_DT_M")
        .withColumn(
            "TOTAL_ACTIVE_TREATMENTS",
            F.when((F.col("TOTAL_TREATMENTS") < 1) | (F.col("TOTAL_TREATMENTS").isNull()), "NO_ACTIVE_TREATMENT")
            .otherwise(
                F.when(F.col("TOTAL_TREATMENTS") == 1, "1_ACTIVE_TREATMENT")
                .otherwise(
                    F.when(F.col("TOTAL_TREATMENTS") < 3, "2_OR_3_ACTIVE_TREATMENTS")
                    .otherwise("3_MORE_ACTIVE_TREATMENTS")
                )
            )
        )
        .withColumn(
            "TOTAL_NEW_TREATMENTS",
            F.when((F.col("TOTAL_NEW_TREATMENTS") < 1) | (F.col("TOTAL_NEW_TREATMENTS").isNull()), "NO_NEW_TREATMENT")
            .otherwise(
                F.when(F.col("TOTAL_NEW_TREATMENTS") == 1, "1_NEW_TREATMENT")
                .otherwise(
                    F.when(F.col("TOTAL_NEW_TREATMENTS") < 3, "2_OR_3_NEW_TREATMENTS")
                    .otherwise("3_MORE_NEW_TREATMENTS")
                )
            )
        )
    )
    df_active = pivoted_counts(df_tx_joined, NPI_COL, "TOTAL_ACTIVE_TREATMENTS",
                               "RUNNING_TOTAL_RX_CLAIMS_PATIENT_WITH_", "", values=PIVOT_VALUES["active_tx"])
    df_new = pivoted_counts(df_tx_joined, NPI_COL, "TOTAL_NEW_TREATMENTS",
                            "RUNNING_TOTAL_RX_CLAIMS_PATIENT_WITH_", "", values=PIVOT_VALUES["new_tx"])
    return df_active.join(df_new, on=NPI_COL, how="outer").na.fill(0)


# ---- Derived running SGLT2_WO_INVOKANA_300MG columns ----
running_sglt2_pairs = [
    ("RUNNING_TOTAL_RX_CLAIMS_SGLT2", "RUNNING_TOTAL_RX_CLAIMS_INVOKANA_300MG", "RUNNING_TOTAL_RX_CLAIMS_SGLT2_WO_INVOKANA_300MG"),
    ("RUNNING_TOTAL_RX_CLAIMS_SGLT2_30_DAYS_SUPPLY", "RUNNING_TOTAL_RX_CLAIMS_INVOKANA_300MG_30_DAYS_SUPPLY", "RUNNING_TOTAL_RX_CLAIMS_SGLT2_WO_INVOKANA_300MG_30_DAYS_SUPPLY"),
    ("RUNNING_TOTAL_RX_CLAIMS_SGLT2_60_DAYS_SUPPLY", "RUNNING_TOTAL_RX_CLAIMS_INVOKANA_300MG_60_DAYS_SUPPLY", "RUNNING_TOTAL_RX_CLAIMS_SGLT2_WO_INVOKANA_300MG_60_DAYS_SUPPLY"),
    ("RUNNING_TOTAL_RX_CLAIMS_SGLT2_90_DAYS_SUPPLY", "RUNNING_TOTAL_RX_CLAIMS_INVOKANA_300MG_90_DAYS_SUPPLY", "RUNNING_TOTAL_RX_CLAIMS_SGLT2_WO_INVOKANA_300MG_90_DAYS_SUPPLY"),
    ("RUNNING_TOTAL_RX_CLAIMS_SGLT2_NOT_30_60_90_DAYS_SUPPLY", "RUNNING_TOTAL_RX_CLAIMS_INVOKANA_300MG_NOT_30_60_90_DAYS_SUPPLY", "RUNNING_TOTAL_RX_CLAIMS_SGLT2_WO_INVOKANA_300MG_NOT_30_60_90_DAYS_SUPPLY"),
    ("RUNNING_TOTAL_RX_CLAIMS_SGLT2_DAYS_SUPPLY_UNKNOWN", "RUNNING_TOTAL_RX_CLAIMS_INVOKANA_300MG_DAYS_SUPPLY_UNKNOWN", "RUNNING_TOTAL_RX_CLAIMS_SGLT2_WO_INVOKANA_300MG_DAYS_SUPPLY_UNKNOWN"),
    ("RUNNING_TOTAL_REJECTED_RX_CLAIMS_SGLT2", "RUNNING_TOTAL_REJECTED_RX_CLAIMS_INVOKANA_300MG", "RUNNING_TOTAL_REJECTED_RX_CLAIMS_SGLT2_WO_INVOKANA_300MG"),
]
for age_bin in [
    "AGE_GROUP_0_10", "AGE_GROUP_10_20", "AGE_GROUP_20_30", "AGE_GROUP_30_40",
    "AGE_GROUP_40_50", "AGE_GROUP_50_60", "AGE_GROUP_60_70", "AGE_GROUP_70_80",
    "AGE_GROUP_ABOVE_80"
]:
    running_sglt2_pairs.append((
        f"RUNNING_TOTAL_RX_CLAIMS_SGLT2_{age_bin}",
        f"RUNNING_TOTAL_RX_CLAIMS_INVOKANA_300MG_{age_bin}",
        f"RUNNING_TOTAL_RX_CLAIMS_SGLT2_WO_INVOKANA_300MG_{age_bin}"
    ))
for gender in ["M", "F", "U"]:
    running_sglt2_pairs.append((
        f"RUNNING_TOTAL_RX_CLAIMS_SGLT2_{gender}",
        f"RUNNING_TOTAL_RX_CLAIMS_INVOKANA_300MG_{gender}",
        f"RUNNING_TOTAL_RX_CLAIMS_SGLT2_WO_INVOKANA_300MG_{gender}"
    ))


def build_running_features(df_full, cb):
    """All RUNNING_ columns for one cohort: full history up to and including the cohort cutoff."""
    cutoff_lit = F.to_date(F.lit(cb["cutoff"].strftime("%Y-%m-%d")))
    df_rx_running = (
        df_full
        .filter(F.col(NPI_COL).isNotNull())
        .filter(F.col(SVC_COL) <= cutoff_lit)
    )
    # Every group's NPIs are a subset of the running-total NPIs, so left joins onto it
    # + one final na.fill(0) give the same result as the original nested outer joins.
    df_running = build_running_total(df_rx_running, NPI_COL)
    for key, pivot_col, prefix, extra_filter, combine_cols in RUNNING_GROUPS:
        df_g = build_running_feature_group(df_rx_running, NPI_COL, pivot_col, prefix,
                                           extra_filter, combine_cols, PIVOT_VALUES[key])
        df_running = df_running.join(df_g, on=NPI_COL, how="left")
    df_running = (df_running
                  .join(build_running_patient_treatment(df_rx_running), on=NPI_COL, how="left")
                  .na.fill(0))

    running_cols = set(df_running.columns)
    for sglt2_col, inv_col, out_col in running_sglt2_pairs:
        if sglt2_col in running_cols and inv_col in running_cols:
            df_running = df_running.withColumn(out_col, F.col(sglt2_col) - F.col(inv_col))
        elif sglt2_col in running_cols:
            df_running = df_running.withColumn(out_col, F.col(sglt2_col))
    return df_running


# ---- Preview on one cohort ----
df_running = build_running_features(df_rx, cb_preview)
print(f"Preview {cb_preview['cohort_id']} | Running universe: {df_running.count():,} NPIs")
print(f"RUNNING_ cols: {len([c for c in df_running.columns if c.startswith('RUNNING_')])}")

In [0]:
# ---- AVG Features (two-stage monthly aggregation + zero-fill) ----
# This overwrites the earlier direct-window df_avg with logic that mirrors the old notebook:
#   1) aggregate to NPI × calendar month
#   2) zero-fill missing months
#   3) sum monthly values and divide by the fixed denominator
# The denominator is N or N+1 depending on AVG_EXTRA_MONTH.


def build_avg_features_two_stage(df_rx_base, df_rx_pat_base, npi_col, svc_dt_col, avg_window_bounds):
    avg_npi_universe = (
        df_rx_base
        .filter(F.col(npi_col).isNotNull())
        .select(npi_col)
        .distinct()
    )

    window_dfs = []

    for w in sorted(avg_window_bounds.keys()):
        bounds = avg_window_bounds[w]
        suffix = f"_IN_LAST_{w}_MONTH"
        denominator = (w + 1) if AVG_EXTRA_MONTH else w

        # Filter to the window first so partial boundary months respect AVG_WINDOW_BOUNDS.
        df_w = (
            filter_to_window(df_rx_base, svc_dt_col, bounds["start"], bounds["end"])
            .withColumn("_SVC_MONTH", assign_cohort_month(F.col(svc_dt_col)))
        )
        df_w_pat = (
            filter_to_window(df_rx_pat_base, svc_dt_col, bounds["start"], bounds["end"])
            .withColumn("_SVC_MONTH", assign_cohort_month(F.col(svc_dt_col)))
        )

        # Monthly metrics (same per-month semantics as the old monthly notebook).
        df_monthly_age = (
            df_w_pat.groupBy(npi_col, "_SVC_MONTH")
            .agg(F.avg("AGE").alias("AVG_PATIENT_AGE_MONTHLY"))
        )

        df_monthly_pat_count = (
            df_w.groupBy(npi_col, "_SVC_MONTH")
            .agg(F.countDistinct("PATIENT_ID").alias("TOTAL_PATIENT_COUNT_MONTHLY"))
        )

        df_monthly_tx_per_pat = (
            df_w.filter("lower(TREATMENT_CATEGORY) NOT LIKE '%other%'")
            .groupBy(npi_col, "_SVC_MONTH", "PATIENT_ID")
            .agg(F.countDistinct("TREATMENT_CATEGORY").alias("tx_count"))
            .groupBy(npi_col, "_SVC_MONTH")
            .agg(
                F.sum("tx_count").alias("_total_tx"),
                F.countDistinct("PATIENT_ID").alias("_total_pat")
            )
            .withColumn(
                "AVG_TREATMENT_PER_PATIENT_MONTHLY",
                F.when(F.col("_total_pat") != 0, F.col("_total_tx") / F.col("_total_pat"))
                .otherwise(F.lit(0.0))
            )
            .drop("_total_tx", "_total_pat")
        )

        df_monthly = (
            df_monthly_age
            .join(df_monthly_pat_count, on=[npi_col, "_SVC_MONTH"], how="outer")
            .join(df_monthly_tx_per_pat, on=[npi_col, "_SVC_MONTH"], how="outer")
        )

        # Build the month spine so empty months count as zero.
        first_month = (datetime.strptime(bounds["start"], "%Y-%m-%d") + timedelta(days=1)).strftime("%Y-%m-01")
        last_month = datetime.strptime(bounds["end"], "%Y-%m-%d").strftime("%Y-%m-01")

        month_spine = (
            spark.range(1)
            .select(
                F.explode(
                    F.sequence(
                        F.to_date(F.lit(first_month)),
                        F.to_date(F.lit(last_month)),
                        F.expr("interval 1 month")
                    )
                ).alias("_SVC_MONTH")
            )
        )

        df_monthly_filled = (
            avg_npi_universe
            .crossJoin(month_spine)
            .join(df_monthly, on=[npi_col, "_SVC_MONTH"], how="left")
            .na.fill({
                "AVG_PATIENT_AGE_MONTHLY": 0.0,
                "TOTAL_PATIENT_COUNT_MONTHLY": 0.0,
                "AVG_TREATMENT_PER_PATIENT_MONTHLY": 0.0,
            })
        )

        df_window = (
            df_monthly_filled
            .groupBy(npi_col)
            .agg(
                (F.sum("AVG_PATIENT_AGE_MONTHLY") / F.lit(float(denominator))).alias(f"AVG_PATIENT_AGE{suffix}"),
                (F.sum("TOTAL_PATIENT_COUNT_MONTHLY") / F.lit(float(denominator))).alias(f"TOTAL_PATIENT_COUNT{suffix}"),
                (F.sum("AVG_TREATMENT_PER_PATIENT_MONTHLY") / F.lit(float(denominator))).alias(f"AVG_TREATMENT_PER_PATIENT{suffix}")
            )
        )

        window_dfs.append(df_window)

    result = window_dfs[0]
    for df_next in window_dfs[1:]:
        result = result.join(df_next, on=npi_col, how="outer")

    return result.na.fill(0)


# ---- Preview on one cohort ----
# df_rx_c already has AGE (add_rx_derived_cols), so it is both the claims and the patient-attribute base.
df_avg = build_avg_features_two_stage(df_rx_c, df_rx_c, NPI_COL, SVC_COL, cb_preview["avg_window_bounds"])
print(f"Avg feature cols (two-stage): {len(df_avg.columns) - 1}")

### 2. Combine All Feature Groups + Derived Features

In [0]:
# ---- Combine all feature groups on NPI ----
# Widen NPI universe: include doctors from full history (df_running)
# who have nonzero RUNNING_ values but no claims in the 13-month window.
# Without this, ~1,200 doctors per cohort would get RUNNING_=0 instead of
# their correct historical counts.
def combine_features(npi_universe, df_groups, df_running):
    """Widened NPI universe left-joined to every feature group, then RUNNING_; missing -> 0."""
    npi_universe_wide = npi_universe.union(df_running.select(NPI_COL)).distinct()
    df_spine = npi_universe_wide
    for df_g in df_groups + [df_running]:
        df_spine = df_spine.join(df_g, on=NPI_COL, how="left")
    return df_spine.na.fill(0)


# ---- Preview on one cohort ----
npi_universe_wide = npi_universe.union(df_running.select(NPI_COL)).distinct()
print(f"NPI universe widened: {npi_universe.count():,} → {npi_universe_wide.count():,} "
      f"(+{npi_universe_wide.count() - npi_universe.count():,} from older history)")

df_spine = combine_features(npi_universe, [
    df_total,
    df_category,
    df_brand,
    df_rejection,
    df_duration,
    df_refill,
    df_payer,
    df_patient,
    df_patient_treatment,
    df_avg,
], df_running)

print(f"Combined spine: {df_spine.count():,} NPIs, {len(df_spine.columns)} columns")

In [0]:
# ---- SGLT2 Without Invokana 300MG (same derived columns as original) ----
# These subtract Invokana 300MG from SGLT2 totals.
# We generate them for each window that has both columns.

sglt2_wo_pairs = [
    # (SGLT2 col pattern, INVOKANA col pattern, output pattern)
    ("TOTAL_RX_CLAIMS_SGLT2", "TOTAL_RX_CLAIMS_INVOKANA_300MG", "TOTAL_RX_CLAIMS_SGLT2_WO_INVOKANA_300MG"),
    ("TOTAL_RX_CLAIMS_SGLT2_30_DAYS_SUPPLY", "TOTAL_RX_CLAIMS_INVOKANA_300MG_30_DAYS_SUPPLY", "TOTAL_RX_CLAIMS_SGLT2_WO_INVOKANA_300MG_30_DAYS_SUPPLY"),
    ("TOTAL_RX_CLAIMS_SGLT2_60_DAYS_SUPPLY", "TOTAL_RX_CLAIMS_INVOKANA_300MG_60_DAYS_SUPPLY", "TOTAL_RX_CLAIMS_SGLT2_WO_INVOKANA_300MG_60_DAYS_SUPPLY"),
    ("TOTAL_RX_CLAIMS_SGLT2_90_DAYS_SUPPLY", "TOTAL_RX_CLAIMS_INVOKANA_300MG_90_DAYS_SUPPLY", "TOTAL_RX_CLAIMS_SGLT2_WO_INVOKANA_300MG_90_DAYS_SUPPLY"),
    ("TOTAL_REJECTED_RX_CLAIMS_SGLT2", "TOTAL_REJECTED_RX_CLAIMS_INVOKANA_300MG", "TOTAL_REJECTED_RX_CLAIMS_SGLT2_WO_INVOKANA_300MG"),
]

# Also age group and gender variants
for age_bin in ["AGE_GROUP_0_10", "AGE_GROUP_10_20", "AGE_GROUP_20_30", "AGE_GROUP_30_40",
                "AGE_GROUP_40_50", "AGE_GROUP_50_60", "AGE_GROUP_60_70", "AGE_GROUP_70_80",
                "AGE_GROUP_ABOVE_80"]:
    sglt2_wo_pairs.append((
        f"TOTAL_RX_CLAIMS_SGLT2_{age_bin}",
        f"TOTAL_RX_CLAIMS_INVOKANA_300MG_{age_bin}",
        f"TOTAL_RX_CLAIMS_SGLT2_WO_INVOKANA_300MG_{age_bin}"
    ))

for gender in ["M", "F", "U"]:
    sglt2_wo_pairs.append((
        f"TOTAL_RX_CLAIMS_SGLT2_{gender}",
        f"TOTAL_RX_CLAIMS_INVOKANA_300MG_{gender}",
        f"TOTAL_RX_CLAIMS_SGLT2_WO_INVOKANA_300MG_{gender}"
    ))

def add_sglt2_wo(df_spine):
    # All columns added in one withColumns (same columns and order as the old withColumn loop,
    # but a flat plan instead of ~85 nested projections).
    spine_cols = set(df_spine.columns)
    new_cols = {}
    for sglt2_pat, inv_pat, out_pat in sglt2_wo_pairs:
        for w in LOOKBACK_WINDOWS:
            sfx = f"_IN_LAST_{w}_MONTH"
            sglt2_col = f"{sglt2_pat}{sfx}"
            inv_col = f"{inv_pat}{sfx}"
            out_col = f"{out_pat}{sfx}"
            if sglt2_col in spine_cols and inv_col in spine_cols:
                new_cols[out_col] = F.col(sglt2_col) - F.col(inv_col)
            elif sglt2_col in spine_cols:
                # Invokana col missing (no data) — SGLT2 WO = SGLT2
                new_cols[out_col] = F.col(sglt2_col)
    return df_spine.withColumns(new_cols) if new_cols else df_spine


# ---- Preview on one cohort ----
df_spine = add_sglt2_wo(df_spine)
print(f"After SGLT2_WO: {len(df_spine.columns)} columns")

In [0]:
# ---- PCT Features: each feature / total claims in the same window ----
# AVG-family features are averages, not claim counts: the 3.0 original has no PCT_ for them
_NO_PCT_PREFIXES = ("AVG_PATIENT_AGE", "TOTAL_PATIENT_COUNT", "AVG_TREATMENT_PER_PATIENT", "TOTAL_TREATMENT_COUNT")

def add_pct_features(df_spine):
    spine_cols = df_spine.columns
    pct_exprs = []
    for w in LOOKBACK_WINDOWS:
        sfx = f"_IN_LAST_{w}_MONTH"
        denom_col = f"TOTAL_RX_CLAIMS{sfx}"
        if denom_col not in spine_cols:
            continue
        for c in spine_cols:
            if c.endswith(sfx) and c != denom_col and c != NPI_COL and not c.startswith(_NO_PCT_PREFIXES):
                pct_name = f"PCT_{c}"
                pct_exprs.append(
                    F.when(F.col(denom_col) != 0, F.col(c) / F.col(denom_col))
                    .otherwise(F.lit(0.0))
                    .alias(pct_name)
                )
    return df_spine.select("*", *pct_exprs)


# ---- Preview on one cohort ----
df_spine = add_pct_features(df_spine)
print(f"After PCT: {len(df_spine.columns)} columns")

In [0]:
# ---- Difference features: _IN_LAST_3_6_MONTH = last_6 - last_3 ----
# ---- Ratio features: prior_3 / last_3 ----
# These measure the "prior 3 months" (months 4-6 back) vs the most recent 3.

from pyspark.sql.functions import col, when, lit

def safe_div(numerator_col, denominator_col):
    return when(col(denominator_col) != 0, col(numerator_col) / col(denominator_col)).otherwise(0.0)

# Treatment categories and brands for diff/ratio
diff_ratio_features = [
    "TOTAL_RX_CLAIMS_ACE_OR_ARB",
    "TOTAL_RX_CLAIMS_CALCIUM_CHANNEL_OR_BETA_BLOCKER",
    "TOTAL_RX_CLAIMS_COMBO_DRUG",
    "TOTAL_RX_CLAIMS_DIURETIC_NON_MRA",
    "TOTAL_RX_CLAIMS_INSULIN",
    "TOTAL_RX_CLAIMS_MRA",
    "TOTAL_RX_CLAIMS_SGLT2",
    "TOTAL_RX_CLAIMS_METFORMIN",
    "TOTAL_RX_CLAIMS_FARXIGA",
    "TOTAL_RX_CLAIMS_JARDIANCE",
    "TOTAL_RX_CLAIMS_SPIRONOLACTONE_MRA",
    "TOTAL_RX_CLAIMS",
]

def add_diff_ratio(df_spine):
    # Only create diff/ratio if both 3m and 6m windows exist
    if 3 in LOOKBACK_WINDOWS and 6 in LOOKBACK_WINDOWS:
        spine_cols = set(df_spine.columns)
        new_cols = {}   # one withColumns instead of a withColumn loop (same columns, same order)

        # Same as the 3.0 original: when the feature's whole group (all claims for the category
        # features and TOTAL_RX_CLAIMS, brand-filtered claims for the brands) had no claims in the
        # last 3 months, the original's last_3 was null, so last_6 - last_3 was null -> filled 0.
        _brand_feats = {f"TOTAL_RX_CLAIMS_{clean_col_name(v)}" for v in PIVOT_VALUES["brand"]}
        _brand_3m = [f"{b}_IN_LAST_3_MONTH" for b in sorted(_brand_feats) if f"{b}_IN_LAST_3_MONTH" in spine_cols]
        _guard = {
            "all":   col("TOTAL_RX_CLAIMS_IN_LAST_3_MONTH") > 0,
            "brand": (builtins.sum((col(c) for c in _brand_3m), lit(0)) > 0) if _brand_3m else lit(False),
        }
        for feat in diff_ratio_features:
            col_3m = f"{feat}_IN_LAST_3_MONTH"
            col_6m = f"{feat}_IN_LAST_6_MONTH"
            diff_col = f"{feat}_IN_LAST_3_6_MONTH"
            ratio_col = feat.replace("TOTAL_RX_CLAIMS", "RATIO_RX_CLAIMS") + "_IN_PRIOR_3_LAST_3_MONTH"
            if col_3m in spine_cols and col_6m in spine_cols:
                # Difference: prior 3 months = last 6 - last 3
                _g = _guard["brand" if feat in _brand_feats else "all"]
                new_cols[diff_col] = when(_g, col(col_6m) - col(col_3m)).otherwise(lit(0))
                # Ratio: prior 3 / last 3 (= safe_div(diff_col, col_3m), written inline)
                new_cols[ratio_col] = when(col(col_3m) != 0,
                                           (col(col_6m) - col(col_3m)) / col(col_3m)).otherwise(0.0)
        if new_cols:
            df_spine = df_spine.withColumns(new_cols)
    return df_spine


# ---- Preview on one cohort ----
df_spine = add_diff_ratio(df_spine)
print(f"After Diff/Ratio: {len(df_spine.columns)} columns")

In [0]:
# ---- Add cohort metadata ----
def add_cohort_metadata(df_spine, cb):
    return df_spine.withColumns({
        "COHORT_ID":    F.lit(cb["cohort_id"]),
        "COHORT_MONTH": F.lit(cb["cohort_month"]),
        "OFFSET_DAYS":  F.lit(cb["offset_days"]),
        "CUTOFF_DATE":  F.lit(cb["cutoff"].strftime("%Y-%m-%d")),
        "SVC_DT_M":     F.to_timestamp(F.lit(f"{cb['cohort_month']}-01")),
        "PULL_DT":      F.current_date(),
    })


# ---- Preview on one cohort ----
claim_rx_monthly = add_cohort_metadata(df_spine, cb_preview)

# Column count is metadata-only (no Spark action needed)
print(f"Final output columns: {len(claim_rx_monthly.columns)}")
# NOTE: count() and display() removed — each forces a full pipeline re-evaluation
# without .cache(). Row count is verified after the write (cell 26).

### 2b. Fast Cohort Engine (used by section 3)

The step-by-step functions above re-read `Rx_01_new` (31 GB, not clustered) about **140 times per cohort** (17 feature groups x 5 windows, totals, patient treatment, AVG, RUNNING_). That is why one cohort took over 10 minutes even for 10 HCPs.

The fast engine computes the **same features** in two steps:
1. **Once per run**: every claim is turned into its feature keys (e.g. `category = SGLT2`, `payer_brand = FARXIGA_Commercial`) and counted per **NPI x cohort month x key**. A second small table holds the monthly AVG inputs. Both are kept in memory (validation) or saved to a staging Delta table (production).
2. **Per cohort**: one pass over that compact table. Each month is tagged with the windows it falls in (1, 3, 6, 12 months, RUNNING_), then a single pivot builds all count columns.

Column names, column order and values are the same as the step-by-step functions (the cell after next checks this against partitions written by the old code). The windows line up exactly with cohort months (checked for every cohort), so counting by month gives the same result as filtering by date.

In [0]:
# ============================================================
# FAST COHORT ENGINE — read the claims once, one aggregation per cohort
# ============================================================
import time
from dateutil.relativedelta import relativedelta

_SEP = "\u0001"   # separator inside pivot keys (never appears in the data)
_STAGE_PREFIX = _NOTEBOOK_OUTPUT + "claim_rx_monthly_cohort_stage_"


# materialize(df, name) comes from 00_cohort_config (in memory in validation, staging Delta table
# _STAGE_PREFIX + name in production).


# ---------- Feature groups: same definitions and order as the step-by-step functions ----------
# (PIVOT_VALUES key, column prefix, claim condition, pivot value)
_IS_BRAND = F.coalesce(F.expr(BRAND_FILTER), F.lit(False))
_IS_REJ   = F.coalesce(F.expr("REJECTION_FLAG = 1"), F.lit(False))

def _cw(*cols):
    """Combined pivot value, same as build_feature_group (concat_ws skips nulls)."""
    return F.concat_ws("_", *[F.col(c) for c in cols])

_TC, _CAT = "TREATMENT_CATEGORY", "Category"
CLAIM_GROUPS = [
    ("category",        "TOTAL_RX_CLAIMS_",          None,                F.col(_TC)),
    ("brand",           "TOTAL_RX_CLAIMS_",          _IS_BRAND,           F.col(_CAT)),
    ("rej_category",    "TOTAL_REJECTED_RX_CLAIMS_", _IS_REJ,             F.col(_TC)),
    ("rej_brand",       "TOTAL_REJECTED_RX_CLAIMS_", _IS_REJ & _IS_BRAND, F.col(_CAT)),
    ("dur_category",    "TOTAL_RX_CLAIMS_",          None,                _cw(_TC, "DAYS_SUPPLY_BIN")),
    ("dur_brand",       "TOTAL_RX_CLAIMS_",          _IS_BRAND,           _cw(_CAT, "DAYS_SUPPLY_BIN")),
    ("refill_alone",    "TOTAL_RX_CLAIMS_",          None,                F.col("REFILL_FLAG")),
    ("refill_brand",    "TOTAL_RX_CLAIMS_",          _IS_BRAND,           _cw(_CAT, "REFILL_FLAG")),
    ("payer_alone",     "TOTAL_RX_CLAIMS_",          None,                F.col("METHOD_OF_PAYMENT")),
    ("payer_category",  "TOTAL_RX_CLAIMS_",          None,                _cw(_TC, "METHOD_OF_PAYMENT")),
    ("payer_brand",     "TOTAL_RX_CLAIMS_",          _IS_BRAND,           _cw(_CAT, "METHOD_OF_PAYMENT")),
    ("age_alone",       "TOTAL_RX_CLAIMS_",          None,                F.col("AGE_BIN")),
    ("age_category",    "TOTAL_RX_CLAIMS_",          None,                _cw(_TC, "AGE_BIN")),
    ("age_brand",       "TOTAL_RX_CLAIMS_",          _IS_BRAND,           _cw(_CAT, "AGE_BIN")),
    ("gender_alone",    "TOTAL_RX_CLAIMS_",          None,                F.col("PATIENT_GENDER")),
    ("gender_category", "TOTAL_RX_CLAIMS_",          None,                _cw(_TC, "PATIENT_GENDER")),
    ("gender_brand",    "TOTAL_RX_CLAIMS_",          _IS_BRAND,           _cw(_CAT, "PATIENT_GENDER")),
]
PT_GROUPS = [("active_tx", "TOTAL_RX_CLAIMS_PATIENT_WITH_"), ("new_tx", "TOTAL_RX_CLAIMS_PATIENT_WITH_")]


# ---------- Step 1a: claims -> (NPI, cohort month, feature key, N) ----------
def build_feature_long(df, offset_days, pat_tx):
    d = (df.filter(F.col(NPI_COL).isNotNull())
           .withColumn("_CM", assign_cohort_month(F.col(SVC_COL), offset_days)))

    keys = [F.lit("total" + _SEP)]
    for g, _prefix, cond, val in CLAIM_GROUPS:
        k = F.concat(F.lit(g + _SEP), val)          # null value -> null key -> dropped (like pivot)
        keys.append(k if cond is None else F.when(cond, k))
    claims = (d.select(NPI_COL, "_CM", F.explode(F.array(*keys)).alias("KEY"))
               .filter(F.col("KEY").isNotNull()))

    # Patient treatment buckets: same left join as build_patient_treatment_features, on the claim's
    # month at the COHORT's offset (fix 2026-10-22: was always the global OFFSET_DAY).
    _pat = pat_tx.select(
        F.col("PATIENT_ID").alias("_P"), F.to_date("SVC_DT_M").alias("_PM"),
        "TOTAL_TREATMENTS", "TOTAL_NEW_TREATMENTS")
    tt, tn = F.col("TOTAL_TREATMENTS"), F.col("TOTAL_NEW_TREATMENTS")
    active = (F.when((tt < 1) | tt.isNull(), "NO_ACTIVE_TREATMENT").when(tt == 1, "1_ACTIVE_TREATMENT")
               .when(tt < 3, "2_OR_3_ACTIVE_TREATMENTS").otherwise("3_MORE_ACTIVE_TREATMENTS"))
    new = (F.when((tn < 1) | tn.isNull(), "NO_NEW_TREATMENT").when(tn == 1, "1_NEW_TREATMENT")
            .when(tn < 3, "2_OR_3_NEW_TREATMENTS").otherwise("3_MORE_NEW_TREATMENTS"))
    pt = (d.join(_pat, (F.col("PATIENT_ID") == F.col("_P")) & (F.col("_CM") == F.col("_PM")), "left")
           .select(NPI_COL, "_CM", F.explode(F.array(
               F.concat(F.lit("active_tx" + _SEP), active),
               F.concat(F.lit("new_tx" + _SEP), new))).alias("KEY")))

    return (claims.unionByName(pt)
            .groupBy(NPI_COL, "_CM", "KEY").agg(F.count(F.lit(1)).alias("N")))


# ---------- Step 1b: monthly AVG inputs (per-month logic of the original 3.0 notebook) ----------
# Fixes 2026-10-22: months use the cohort's offset (was the global OFFSET_DAY), and
# TOTAL_PATIENT_COUNT counts distinct patients on non-'other' claims only, as in the original
# (df_avg_final in 01_claim_rx_monthly); 3.1 had counted all patients.
def build_monthly_avg(df, offset_days):
    d = (df.filter(F.col(NPI_COL).isNotNull())
           .withColumn("_CM", assign_cohort_month(F.col(SVC_COL), offset_days)))
    k = [NPI_COL, "_CM"]
    age = d.groupBy(*k).agg(F.avg("AGE").alias("AVG_PATIENT_AGE_MONTHLY"))
    tx = (d.filter("lower(TREATMENT_CATEGORY) NOT LIKE '%other%'")
            .groupBy(*k, "PATIENT_ID").agg(F.countDistinct("TREATMENT_CATEGORY").alias("tx_count"))
            .groupBy(*k).agg(F.sum("tx_count").alias("_t"), F.countDistinct("PATIENT_ID").alias("_p"))
            .withColumn("AVG_TREATMENT_PER_PATIENT_MONTHLY",
                        F.when(F.col("_p") != 0, F.col("_t") / F.col("_p")).otherwise(F.lit(0.0)))
            .withColumnRenamed("_p", "TOTAL_PATIENT_COUNT_MONTHLY")
            .withColumnRenamed("_t", "TOTAL_TREATMENT_COUNT_MONTHLY"))   # kept: original TOTAL_TREATMENT_COUNT
    return (age.join(tx, k, "outer")
            .na.fill({"AVG_PATIENT_AGE_MONTHLY": 0.0, "TOTAL_PATIENT_COUNT_MONTHLY": 0.0,
                      "AVG_TREATMENT_PER_PATIENT_MONTHLY": 0.0, "TOTAL_TREATMENT_COUNT_MONTHLY": 0}))


# AVG family, same 4 metrics as the original (monthly column, output name, pivot alias).
# Windows: sum of the monthly values / fixed denominator. RUNNING_: sum of the monthly values up to
# the cohort month (original step3_lag_avg running window). Added 2026-10-22: TOTAL_TREATMENT_COUNT
# windows and the 4 RUNNING_ columns, which the original has and 3.1 had dropped.
_AVG_METRICS = [("AVG_PATIENT_AGE_MONTHLY",           "AVG_PATIENT_AGE",           "a"),
                ("TOTAL_PATIENT_COUNT_MONTHLY",       "TOTAL_PATIENT_COUNT",       "p"),
                ("AVG_TREATMENT_PER_PATIENT_MONTHLY", "AVG_TREATMENT_PER_PATIENT", "x"),
                ("TOTAL_TREATMENT_COUNT_MONTHLY",     "TOTAL_TREATMENT_COUNT",     "t")]


# ---------- Output columns in the same order as build_cohort_slow ----------
_W = [str(w) for w in sorted(LOOKBACK_WINDOWS)]

def _out(base, tag):
    return f"RUNNING_{base}" if tag == "RUN" else f"{base}_IN_LAST_{tag}_MONTH"

def _group_specs(tags):
    """(pivot key, output column) for every claim group, window-major inside each group."""
    specs = []
    for g, prefix, _c, _v in CLAIM_GROUPS:
        vals = sorted(PIVOT_VALUES[g])
        for t in tags:
            specs += [(f"{g}{_SEP}{v}{_SEP}{t}", _out(prefix + clean_col_name(v), t)) for v in vals]
    return specs

def _pt_specs(tags):
    specs = []
    for t in tags:
        for g, prefix in PT_GROUPS:
            specs += [(f"{g}{_SEP}{v}{_SEP}{t}", _out(prefix + clean_col_name(v), t))
                      for v in sorted(PIVOT_VALUES[g])]
    return specs

# Running groups are one block each (no windows), so _group_specs(["RUN"]) gives the RUNNING_GROUPS order.
COUNT_SPECS = ([(f"total{_SEP}{_SEP}{t}", _out("TOTAL_RX_CLAIMS", t)) for t in _W]
               + _group_specs(_W) + _pt_specs(_W)
               + [(f"total{_SEP}{_SEP}RUN", "RUNNING_TOTAL_RX_CLAIMS")]
               + _group_specs(["RUN"]) + _pt_specs(["RUN"]))
_N_WINDOWED = len(_W) + len(_group_specs(_W)) + len(_pt_specs(_W))
AVG_COLS = [f"{name}_IN_LAST_{w}_MONTH" for w in _W for _, name, _ in _AVG_METRICS]
AVG_RUN_COLS = [f"RUNNING_{name}" for _, name, _ in _AVG_METRICS]
FAST_COLS = ([c for _, c in COUNT_SPECS[:_N_WINDOWED]] + AVG_COLS
             + [c for _, c in COUNT_SPECS[_N_WINDOWED:]] + AVG_RUN_COLS)
_dups = sorted({c for c in FAST_COLS if FAST_COLS.count(c) > 1})
if _dups:
    raise ValueError(f"Duplicate feature column names: {_dups[:10]}")


# ---------- Window -> cohort-month range (checked: windows must line up with cohort months) ----------
def _cm_of(d, off):
    """Python version of assign_cohort_month."""
    return (d - timedelta(days=off - 1)).replace(day=1) if off > 0 else d.replace(day=1)

def _window_months(b, off):
    s = datetime.strptime(b["start"], "%Y-%m-%d")    # exclusive
    e = datetime.strptime(b["end"], "%Y-%m-%d")      # inclusive
    lo, hi = _cm_of(s + timedelta(days=1), off), _cm_of(e, off)
    if _cm_of(s, off) == lo or _cm_of(e + timedelta(days=1), off) == hi:
        raise ValueError(f"Window {b} does not line up with cohort months at offset {off}")
    return lo.date(), hi.date()


# ---------- Staged inputs (built on first use, one per source x offset) ----------
# "run"  = this run's claims (from DATA_START_DATE) + this run's patient-treatment table.
# The offset-0 parity cell adds "orig" = the original's inputs (full history + 3.0 table).
FAST_SOURCES = {"run": lambda: (df_rx, pat_new_and_total_treatments)}
_FAST_LONG, _FAST_AVG = {}, {}   # (source, offset) -> staged table

def get_feature_long(off, source="run"):
    key = (source, off)
    if key not in _FAST_LONG:
        _t = time.time()
        claims, pat_tx = FAST_SOURCES[source]()
        _FAST_LONG[key] = materialize(build_feature_long(claims, off, pat_tx), f"feature_long_{source}_offset{off}")
        print(f"Feature-key table ({source}, offset {off}) built in {time.time() - _t:,.0f}s")
    return _FAST_LONG[key]

def get_monthly_avg(off, source="run"):
    key = (source, off)
    if key not in _FAST_AVG:
        _t = time.time()
        claims, _ = FAST_SOURCES[source]()
        _FAST_AVG[key] = materialize(build_monthly_avg(claims, off), f"monthly_avg_{source}_offset{off}")
        print(f"Monthly AVG table ({source}, offset {off}) built in {time.time() - _t:,.0f}s")
    return _FAST_AVG[key]


# ---------- Step 2: one cohort ----------
def _build_avg_fast(cb, cm, source="run"):
    off = cb["offset_days"]
    lo = {str(w): _window_months(cb["avg_window_bounds"][w], off)[0] for w in LOOKBACK_WINDOWS}
    aggs = []
    for t in _W:
        w = int(t)
        denom = float(w + 1 if AVG_EXTRA_MONTH else w)
        in_w = F.col("_CM") >= F.lit(lo[t])
        for src, name, _k in _AVG_METRICS:
            aggs.append((F.sum(F.when(in_w, F.col(src)).otherwise(F.lit(0.0))) / F.lit(denom))
                        .alias(f"{name}_IN_LAST_{t}_MONTH"))
    for src, name, _k in _AVG_METRICS:                      # RUNNING_: every month up to the cohort
        aggs.append(F.sum(F.col(src)).alias(f"RUNNING_{name}"))
    return (get_monthly_avg(off, source)
            .filter(F.col("_CM") <= F.lit(cm))
            .groupBy(NPI_COL).agg(*aggs))


def build_cohort_fast(cb, source="run"):
    """All features for one cohort, at the cohort's own offset.
    source: "run" (default) or another key of FAST_SOURCES (the parity cell uses "orig")."""
    off = cb["offset_days"]
    cm  = datetime.strptime(f"{cb['cohort_month']}-01", "%Y-%m-%d").date()
    lo  = {}
    for w in LOOKBACK_WINDOWS:
        _lo, _hi = _window_months(cb["window_bounds"][w], off)
        if _hi != cm:
            raise ValueError(f"{cb['cohort_id']}: window {w} ends in {_hi}, expected {cm}")
        lo[str(w)] = _lo

    # Every month up to the cohort counts for RUNNING_; recent months also count for each window.
    tags = [F.when(F.col("_CM") >= F.lit(lo[t]), F.lit(t)) for t in _W] + [F.lit("RUN")]
    df_counts = (
        get_feature_long(off, source)
        .filter(F.col("_CM") <= F.lit(cm))
        .select(NPI_COL, "KEY", "N", F.explode(F.array(*tags)).alias("_TAG"))
        .filter(F.col("_TAG").isNotNull())
        .withColumn("_PK", F.concat(F.col("KEY"), F.lit(_SEP), F.col("_TAG")))
        .groupBy(NPI_COL).pivot("_PK", [k for k, _ in COUNT_SPECS]).agg(F.sum("N"))
        .toDF(NPI_COL, *[c for _, c in COUNT_SPECS])
    )

    df_avg = _build_avg_fast(cb, cm, source)   # any offset, no slow fallback

    df = (df_counts.join(df_avg, on=NPI_COL, how="left").na.fill(0)
          .select(NPI_COL, *FAST_COLS))

    # Derived RUNNING_ SGLT2_WO_INVOKANA_300MG columns (same as build_running_features)
    _cols = set(FAST_COLS)
    _run_sglt2 = {}
    for sglt2_col, inv_col, out_col in running_sglt2_pairs:
        if sglt2_col in _cols and inv_col in _cols:
            _run_sglt2[out_col] = F.col(sglt2_col) - F.col(inv_col)
        elif sglt2_col in _cols:
            _run_sglt2[out_col] = F.col(sglt2_col)
    if _run_sglt2:
        df = df.withColumns(_run_sglt2)

    df = add_diff_ratio(add_pct_features(add_sglt2_wo(df)))
    return add_cohort_metadata(df, cb)


# ---------- Step 2 (all cohorts at once): ONE query for every cohort ----------
# Same features as build_cohort_fast, but Spark plans the ~4,500-column query once instead of once
# per cohort (that planning was ~35 s per cohort even for 10 HCPs). Each month is tagged with every
# (cohort, window) it belongs to through a small broadcast table, then one pivot by NPI x cohort.
from datetime import date
_META_COLS = ["COHORT_ID", "COHORT_MONTH", "OFFSET_DAYS", "CUTOFF_DATE", "SVC_DT_M", "PULL_DT"]
_EARLIEST = date(1900, 1, 1)

def build_all_cohorts_fast(cbs, source="run"):
    """All features for a list of cohort bounds (same offset), same columns/values as
    build_cohort_fast for each cohort, one row per NPI x COHORT_ID."""
    offs = {cb["offset_days"] for cb in cbs}
    if len(offs) != 1:
        raise ValueError(f"All cohorts must share one offset, got {offs}")
    off = offs.pop()

    tag_rows, avg_rows, meta_rows = [], [], []
    for cb in cbs:
        cm = datetime.strptime(f"{cb['cohort_month']}-01", "%Y-%m-%d").date()
        for w in LOOKBACK_WINDOWS:
            _lo, _hi = _window_months(cb["window_bounds"][w], off)
            if _hi != cm:
                raise ValueError(f"{cb['cohort_id']}: window {w} ends in {_hi}, expected {cm}")
            tag_rows.append((cb["cohort_id"], str(w), _lo, cm))
            avg_rows.append((cb["cohort_id"], str(w), _window_months(cb["avg_window_bounds"][w], off)[0], cm))
        tag_rows.append((cb["cohort_id"], "RUN", _EARLIEST, cm))   # RUNNING_: every month up to the cohort
        avg_rows.append((cb["cohort_id"], "RUN", _EARLIEST, cm))
        meta_rows.append((cb["cohort_id"], cb["cohort_month"], int(cb["offset_days"]), cb["cutoff"].strftime("%Y-%m-%d")))

    _tag_schema = "_COHORT string, _TAG string, _LO date, _HI date"
    tags = F.broadcast(spark.createDataFrame(tag_rows, _tag_schema))
    avg_tags = F.broadcast(spark.createDataFrame(avg_rows, _tag_schema))
    _in = lambda: (F.col("_CM") >= F.col("_LO")) & (F.col("_CM") <= F.col("_HI"))

    df_counts = (
        get_feature_long(off, source).join(tags, _in())
        .withColumn("_PK", F.concat(F.col("KEY"), F.lit(_SEP), F.col("_TAG")))
        .groupBy(NPI_COL, "_COHORT").pivot("_PK", [k for k, _ in COUNT_SPECS]).agg(F.sum("N"))
        .toDF(NPI_COL, "_COHORT", *[c for _, c in COUNT_SPECS])
    )

    _a = (get_monthly_avg(off, source).join(avg_tags, _in())
          .groupBy(NPI_COL, "_COHORT").pivot("_TAG", _W + ["RUN"])
          .agg(*[F.sum(src).alias(k) for src, _n, k in _AVG_METRICS]))
    _avg_sel = []
    for t in _W:
        denom = F.lit(float(int(t) + 1 if AVG_EXTRA_MONTH else int(t)))
        for _src, name, k in _AVG_METRICS:
            _avg_sel.append((F.coalesce(F.col(f"{t}_{k}").cast("double"), F.lit(0.0)) / denom)
                            .alias(f"{name}_IN_LAST_{t}_MONTH"))
    for _src, name, k in _AVG_METRICS:
        _avg_sel.append(F.col(f"RUN_{k}").alias(f"RUNNING_{name}"))
    df_avg = _a.select(NPI_COL, "_COHORT", *_avg_sel)

    df = (df_counts.join(df_avg, on=[NPI_COL, "_COHORT"], how="left").na.fill(0)
          .select(NPI_COL, "_COHORT", *FAST_COLS))

    # Derived columns: same functions as the per-cohort engine (row-wise, so cohorts don't mix)
    _cols = set(FAST_COLS)
    _run_sglt2 = {}
    for sglt2_col, inv_col, out_col in running_sglt2_pairs:
        if sglt2_col in _cols and inv_col in _cols:
            _run_sglt2[out_col] = F.col(sglt2_col) - F.col(inv_col)
        elif sglt2_col in _cols:
            _run_sglt2[out_col] = F.col(sglt2_col)
    if _run_sglt2:
        df = df.withColumns(_run_sglt2)
    df = add_diff_ratio(add_pct_features(add_sglt2_wo(df)))

    # Cohort metadata per row (same columns, order and types as add_cohort_metadata)
    meta = (spark.createDataFrame(meta_rows, "COHORT_ID string, COHORT_MONTH string, OFFSET_DAYS int, CUTOFF_DATE string")
            .withColumns({"SVC_DT_M": F.to_timestamp(F.concat(F.col("COHORT_MONTH"), F.lit("-01"))),
                          "PULL_DT": F.current_date()}))
    feat_cols = [c for c in df.columns if c not in (NPI_COL, "_COHORT")]
    return (df.join(F.broadcast(meta), F.col("_COHORT") == F.col("COHORT_ID"))
              .select(NPI_COL, *feat_cols, *_META_COLS))


print(f"Fast engine ready: {len(COUNT_SPECS):,} count columns + {len(AVG_COLS) + len(AVG_RUN_COLS)} AVG columns per cohort")

In [0]:
# ---- Check: the all-cohorts query gives exactly what the per-cohort engine gives ----
# build_cohort_fast matched the step-by-step code (4 cohorts, 0 mismatches, 2026-10-22).
# build_all_cohorts_fast is what cell 31 writes: compare both on the first and last cohort.
# Nothing is written. Takes ~2 min (2 per-cohort builds + 1 all-cohorts build).
import builtins   # round/sum are shadowed by a pyspark.sql.functions star import
import numpy as np
import pandas as pd

_check = [COHORT_LIST[0], COHORT_LIST[-1]]
_t0 = time.time()
_pd_all = build_all_cohorts_fast([COHORT_BOUNDS[m] for m in _check]).toPandas()
print(f"All-cohorts query ({len(_check)} cohorts): {time.time() - _t0:,.0f}s")

_rows = []
for m in _check:
    cb = COHORT_BOUNDS[m]
    _t0 = time.time()
    pd_one = build_cohort_fast(cb).toPandas().set_index(NPI_COL)
    _t_one = time.time() - _t0
    pd_all = _pd_all[_pd_all["COHORT_ID"] == cb["cohort_id"]].set_index(NPI_COL)

    same_order = list(pd_one.columns) == list(pd_all.columns)
    same_types = (pd_one.dtypes.astype(str) == pd_all.dtypes.reindex(pd_one.columns).astype(str)).all()
    cols = [c for c in pd_one.columns if c in pd_all.columns and c != "PULL_DT"]
    num = [c for c in cols if pd.api.types.is_numeric_dtype(pd_one[c])]
    txt = [c for c in cols if c not in num]
    common = sorted(set(pd_one.index) & set(pd_all.index))
    o = pd_one.loc[common, num].astype(float).values
    n = pd_all.loc[common, num].astype(float).values
    bad = ~np.isclose(o, n, rtol=1e-9, atol=1e-9, equal_nan=True)
    txt_bad = int((pd_one.loc[common, txt].astype(str).values != pd_all.loc[common, txt].astype(str).values).sum())
    _rows.append({"cohort": cb["cohort_id"], "npis_one": len(pd_one), "npis_all": len(pd_all),
                  "npis_common": len(common), "columns": len(pd_one.columns),
                  "same_column_order": same_order, "same_types": bool(same_types),
                  "cells_compared": int(bad.size) + len(common) * len(txt),
                  "mismatching_cells": int(bad.sum()) + txt_bad, "per_cohort_seconds": builtins.round(_t_one)})
    if bad.any():
        i, j = np.argwhere(bad)[0]
        print(f"  {cb['cohort_id']} first mismatch: NPI {common[i]} {num[j]} one={o[i, j]} all={n[i, j]}")

df_all_vs_one = pd.DataFrame(_rows)
display(df_all_vs_one)
_ok = (df_all_vs_one["mismatching_cells"].sum() == 0 and df_all_vs_one["same_column_order"].all()
       and df_all_vs_one["same_types"].all()
       and (df_all_vs_one[["npis_one", "npis_all"]].values == df_all_vs_one[["npis_common"]].values).all())
print("PASS — all-cohorts query = per-cohort engine" if _ok else "CHECK MISMATCHES")

In [0]:
# ---- Load written output for tracing ----
# All config variables (COHORT_MONTH, OFFSET_DAYS, CUTOFF_DATE, WINDOW_BOUNDS, etc.)
# are already in scope from 00_cohort_config.
import pandas as pd

_BASE_OUTPUT = _NOTEBOOK_OUTPUT + "claim_rx_monthly_cohort"
claim_rx_monthly = spark.read.format("delta").load(_BASE_OUTPUT).filter(F.col("COHORT_ID") == COHORT_ID)

# Load raw claims for tracing
# Separate name so the pipeline's df_rx (NPI-filtered, derived columns) is not overwritten
_df_rx_trace = spark.read.format("delta").load(_QUERY_OUTPUT + "Rx_01_new"
).withColumn('TREATMENT_CATEGORY', F.regexp_replace('TREATMENT_CATEGORY', r'[\s]*$', ''))

# Pick the HCP with the most claims in 12m
top_npi = (
    claim_rx_monthly
    .select("NPI_NUMBER", "TOTAL_RX_CLAIMS_IN_LAST_12_MONTH")
    .orderBy(F.col("TOTAL_RX_CLAIMS_IN_LAST_12_MONTH").desc())
    .first()[0]
)
print(f"Selected NPI: {top_npi}")
hcp_row = claim_rx_monthly.filter(F.col("NPI_NUMBER") == top_npi)

# ---- Raw claims summary ----
print("\n" + "="*70)
print(f"RAW CLAIMS BREAKDOWN for NPI {top_npi}")
print("="*70)

for w in LOOKBACK_WINDOWS:
    bounds = WINDOW_BOUNDS[w]
    cnt = (
        _df_rx_trace.filter(F.col("NPI_NUMBER") == top_npi)
        .filter((F.col("SVC_DT") > F.lit(bounds["start"])) & (F.col("SVC_DT") <= F.lit(bounds["end"])))
        .count()
    )
    print(f"  Window {w:>2}m ({bounds['start']} to {bounds['end']}): {cnt:,} raw claims")

print("\nTreatment Category mix (last 12m):")
_df_rx_trace.filter(F.col("NPI_NUMBER") == top_npi).groupBy("TREATMENT_CATEGORY").agg(F.count("*").alias("claims")).orderBy(F.desc("claims")).show(truncate=False)

# ---- Transpose final output ----
pdf = hcp_row.toPandas().T
pdf.columns = ["value"]
pdf = pdf[pdf["value"] != 0]

# Group columns by feature type
def classify_col(col_name):
    if col_name.startswith("COHORT") or col_name.startswith("OFFSET") or col_name in ("CUTOFF_DATE", "SVC_DT_M", "PULL_DT", "NPI_NUMBER"):
        return "0_METADATA"
    elif col_name.startswith("PCT_"):
        return "7_PCT"
    elif col_name.startswith("RATIO_"):
        return "8_RATIO"
    elif "_3_6_MONTH" in col_name:
        return "8_DIFF"
    elif col_name.startswith("RUNNING_"):
        return "6_RUNNING"
    elif col_name.startswith("AVG_") or col_name.startswith("TOTAL_PATIENT_COUNT"):
        return "5_AVG"
    elif "REJECTED" in col_name:
        return "2_REJECTION"
    elif "DAYS_SUPPLY" in col_name:
        return "3_DURATION"
    elif "REFILL" in col_name or "NEW_PRESCRIPTION" in col_name:
        return "3_REFILL"
    elif "METHOD_OF_PAYMENT" in col_name:
        return "4_PAYER"
    elif "AGE_GROUP" in col_name or "PATIENT_GENDER" in col_name:
        return "4_PATIENT_ATTR"
    elif "ACTIVE_TREATMENT" in col_name or "NEW_TREATMENT" in col_name:
        return "5_PATIENT_TX"
    elif "SGLT2_WO" in col_name:
        return "2_SGLT2_WO"
    elif col_name.startswith("TOTAL_RX_CLAIMS_IN_LAST"):
        return "1_TOTAL"
    else:
        return "1_TREATMENT_BRAND"

pdf["group"] = [classify_col(idx) for idx in pdf.index]

print("\n" + "="*70)
print(f"FINAL OUTPUT — NON-ZERO FEATURES for NPI {top_npi}")
print(f"Cohort: {COHORT_MONTH} | Offset: {OFFSET_DAYS}d | Cutoff: {CUTOFF_DATE.date()}")
print(f"Total columns: {len(claim_rx_monthly.columns):,} | Non-zero: {len(pdf):,}")
print("="*70)

for group_name in sorted(pdf["group"].unique()):
    group_df = pdf[pdf["group"] == group_name].drop(columns=["group"])
    label = group_name.split("_", 1)[1]
    print(f"\n--- {label} ({len(group_df)} features) ---")
    for idx, row in group_df.iterrows():
        val = row["value"]
        if isinstance(val, float):
            print(f"  {idx:<80s} {val:>12.4f}")
        else:
            print(f"  {idx:<80s} {val:>12}")

### 3. Build and Write All Cohorts
Runs the same functions as the preview cells (`cohort_slice` → base groups → rejection/duration → refill/payer → patient attributes → patient treatment → AVG → RUNNING_ → combine → SGLT2_WO → PCT → diff/ratio → metadata) for every cohort in `COHORT_LIST`, writing one `COHORT_ID` partition per cohort. Partitions already written are kept if a later cohort fails.

In [0]:
# ---- ONE-TIME RESET (2026-10-23): remove the old tables so the 9-month columns are gone ----
# The write uses mergeSchema, so columns dropped from the code stay in the table as nulls.
# Deleting the table forces it to be recreated with only the current columns.
# Safe: only our 3.1 tables are touched; Wenjie's 3.0 claim_rx_monthly is not affected.
# After one successful Run all, COMMENT OUT or DELETE this cell.
import shutil, os

_RESET_PATHS = [
    _NOTEBOOK_OUTPUT + "claim_rx_monthly_cohort",
    _NOTEBOOK_OUTPUT + "provider_claim_rx_monthly_cohort",
]
for _p in _RESET_PATHS:
    if os.path.exists(_p.replace("/Volumes/", "/Volumes/")):
        try:
            dbutils.fs.rm(_p.replace("/Volumes/", "dbfs:/Volumes/"), recurse=True)
            print(f"DELETED: {_p}")
        except Exception as e:
            print(f"Could not delete {_p}: {e}")
    else:
        print(f"Not found (already clean): {_p}")

In [0]:
# ---- Write output ----
# Partitioned by COHORT_ID so multiple cohorts can coexist.
OUTPUT_PATH = _NOTEBOOK_OUTPUT + "claim_rx_monthly_cohort"

import time

def build_cohort_slow(cb, df_base=None):
    """Original step-by-step version, kept for reference: ~140 reads of Rx_01_new per cohort.
    All feature steps for one cohort (same functions as the preview cells above).
    df_base: claims the windows are sliced from (default df_rx_filtered); RUNNING_ always uses df_rx."""
    df_c = cohort_slice(df_rx_filtered if df_base is None else df_base, cb)
    npi_universe, df_total, df_category, df_brand = build_base_groups(df_c, cb)
    df_rejection, df_duration = build_rejection_duration(df_c, cb)
    df_refill, df_payer = build_refill_payer(df_c, cb)
    df_patient = build_patient_attributes(df_c, cb)
    df_patient_treatment = build_patient_treatment_features(
        df_c, pat_new_and_total_treatments, NPI_COL, SVC_COL, cb["window_bounds"], BRAND_FILTER)
    df_avg = build_avg_features_two_stage(df_c, df_c, NPI_COL, SVC_COL, cb["avg_window_bounds"])
    df_running = build_running_features(df_rx, cb)
    df_spine = combine_features(npi_universe, [
        df_total, df_category, df_brand, df_rejection, df_duration, df_refill,
        df_payer, df_patient, df_patient_treatment, df_avg,
    ], df_running)
    df_spine = add_diff_ratio(add_pct_features(add_sglt2_wo(df_spine)))
    return add_cohort_metadata(df_spine, cb)


# Fast engine (section 2b): same output, claims are read once per run instead of ~140x per cohort.
# Switch back with build_cohort = build_cohort_slow if ever needed.
build_cohort = build_cohort_fast

# All cohorts in ONE query and ONE write (build_all_cohorts_fast, section 2b): Spark plans the
# ~4,500-column query once instead of once per cohort. Only this run's COHORT_ID partitions are
# replaced; other partitions in the table (e.g. other offsets) are kept.
_t_all = time.time()
_ids_sql = ", ".join(f"'{COHORT_BOUNDS[m]['cohort_id']}'" for m in COHORT_LIST)
_df_out = build_all_cohorts_fast([COHORT_BOUNDS[m] for m in COHORT_LIST])
OUTPUT_COLS = _df_out.columns          # columns of this run (used by the checks below)
_df_out.write \
    .format("delta") \
    .mode("overwrite") \
    .option("mergeSchema", "true") \
    .option("replaceWhere", f"COHORT_ID IN ({_ids_sql})") \
    .partitionBy("COHORT_ID") \
    .save(OUTPUT_PATH)

print(f"{len(COHORT_LIST)} cohorts ({COHORT_BOUNDS[COHORT_LIST[0]]['cohort_id']} .. "
      f"{COHORT_BOUNDS[COHORT_LIST[-1]]['cohort_id']}) written to {OUTPUT_PATH} "
      f"in {(time.time() - _t_all) / 60:,.1f} min")

In [0]:
# Read the written table directly (independent of the preview variables)
df_all_cohorts = spark.read.format("delta").load(OUTPUT_PATH)

# ---- Completeness checks ----
_ids_in_table  = sorted(r[0] for r in df_all_cohorts.select("COHORT_ID").distinct().collect())
_missing       = sorted(set(COHORT_IDS) - set(_ids_in_table))
_extra         = sorted(set(_ids_in_table) - set(COHORT_IDS))
# Columns of this run (the table may hold extra columns from earlier runs via mergeSchema)
_run_cols      = OUTPUT_COLS          # set by the write cell
df_all_cohorts = df_all_cohorts.filter(F.col("COHORT_ID").isin(COHORT_IDS)).select(*_run_cols)
_n_rows        = df_all_cohorts.count()
_n_keys        = df_all_cohorts.select(NPI_COL, "COHORT_ID").distinct().count()
_n_npis        = df_all_cohorts.select(NPI_COL).distinct().count()
# One flat aggregate per column, summed in Python (a single a+b+c+... expression over ~4,500
# columns is too deep for the Spark Connect client -> RecursionError)
_null_row      = df_all_cohorts.agg(
    *[F.sum(F.col(c).isNull().cast("int")).alias(f"_n{i}") for i, c in enumerate(_run_cols)]).first()
_null_cells    = builtins.sum(v or 0 for v in _null_row)

print(f"Cohorts expected: {len(COHORT_IDS)} | in table: {len(_ids_in_table)} | "
      f"missing: {_missing or 'none'} | extra (not in this run): {_extra or 'none'}")
print(f"Rows: {_n_rows:,} | distinct (NPI, COHORT_ID): {_n_keys:,} | duplicates: {_n_rows - _n_keys} | "
      f"NPIs: {_n_npis} | Columns: {len(_run_cols):,} | null cells: {_null_cells:,}")
display(df_all_cohorts.groupBy("COHORT_ID", "CUTOFF_DATE").count().orderBy("COHORT_ID"))

# ---- Full table: every NPI x cohort, all columns (keys and dates first) ----
_key_cols  = [NPI_COL, "COHORT_ID", "COHORT_MONTH", "OFFSET_DAYS", "CUTOFF_DATE", "SVC_DT_M", "PULL_DT"]
_feat_cols = [c for c in _run_cols if c not in _key_cols]
display(df_all_cohorts.select(*_key_cols, *_feat_cols).orderBy(NPI_COL, "COHORT_ID"))

In [0]:
# ---- Provider-level output (same mapping as NB02-NB04: NPI -> PROVIDER_ID, NPI dropped) ----
# The original maps NPI to the provider at the end (cells 73-74). Separate table, so the NPI-level
# claim_rx_monthly_cohort above stays as it is. Wenjie's 3.0 tables are NOT touched.
hcp_npi_mapping = spark.read.format("delta").load(_QUERY_OUTPUT + HCP_NPI_MAPPING_TABLE)
hcp_npi_mapping_df = hcp_npi_mapping.select("NPI_NUMBER", "PROVIDER_ID").distinct()

provider_claim_rx_monthly = (
    spark.read.format("delta").load(OUTPUT_PATH)
    .filter(F.col("COHORT_ID").isin(COHORT_IDS))
    .join(hcp_npi_mapping_df, on="NPI_NUMBER", how="left")
    .drop("NPI_NUMBER")
)

PROVIDER_OUTPUT_PATH = _NOTEBOOK_OUTPUT + "provider_claim_rx_monthly_cohort"
provider_claim_rx_monthly.write.format("delta") \
    .mode("overwrite") \
    .option("mergeSchema", "true") \
    .option("replaceWhere", "COHORT_ID IN (" + ", ".join(f"'{c}'" for c in COHORT_IDS) + ")") \
    .partitionBy("COHORT_ID") \
    .save(PROVIDER_OUTPUT_PATH)

_prov = spark.read.format("delta").load(PROVIDER_OUTPUT_PATH).filter(F.col("COHORT_ID").isin(COHORT_IDS))
print(f"Written to {PROVIDER_OUTPUT_PATH} ({len(COHORT_IDS)} cohorts) | rows: {_prov.count():,} | "
      f"columns: {len(_prov.columns):,} | "
      f"null PROVIDER_ID: {_prov.filter(F.col('PROVIDER_ID').isNull()).count():,} | "
      f"duplicate (PROVIDER_ID, COHORT_ID): {_prov.count() - _prov.select('PROVIDER_ID', 'COHORT_ID').distinct().count():,}")

In [0]:
# ---- Validation 2: HCP trace — raw Rx claims recounted in pandas vs written features, every cohort ----
import pandas as pd

TRACE_NPI = HCP_TRACE_NPIS["rx"]   # fixed trace HCP (00_cohort_config)

_BRANDS = HCP_BRANDS   # = BRAND_FILTER (00_cohort_config)
_raw = (df_rx.filter(F.col(NPI_COL) == TRACE_NPI)
        .select(SVC_COL, "TREATMENT_CATEGORY", "Category", "REJECTION_FLAG").toPandas())
_raw.columns = [SVC_COL, "TREATMENT_CATEGORY", "Category", "REJECTION_FLAG"]
_raw[SVC_COL] = pd.to_datetime(_raw[SVC_COL])
_raw["_REJ"]   = _raw["REJECTION_FLAG"].apply(lambda x: x is not None and not pd.isna(x) and int(x) == 1)
_raw["_BRAND"] = _raw["Category"].isin(_BRANDS)

_out = (spark.read.format("delta").load(OUTPUT_PATH)
        .filter((F.col(NPI_COL) == TRACE_NPI) & F.col("COHORT_ID").isin(COHORT_IDS))
        .toPandas().set_index("COHORT_ID"))

def _v(d, k):
    x = d.get(k, 0)
    return 0 if pd.isna(x) else int(x)

rows, bad = [], []
for m in COHORT_LIST:
    cb  = COHORT_BOUNDS[m]
    cid = cb["cohort_id"]
    feat = _out.loc[cid].to_dict() if cid in _out.index else {}
    row = {"COHORT_ID": cid, "CUTOFF_DATE": f"{cb['cutoff']:%Y-%m-%d}"}
    n_checks = n_bad = 0
    for w in LOOKBACK_WINDOWS:
        b = cb["window_bounds"][w]
        sel = _raw[(_raw[SVC_COL] > pd.Timestamp(b["start"])) & (_raw[SVC_COL] <= pd.Timestamp(b["end"]))]
        sfx = f"_IN_LAST_{w}_MONTH"
        expected = {f"TOTAL_RX_CLAIMS{sfx}": len(sel)}
        for v in PIVOT_VALUES["category"]:
            expected[f"TOTAL_RX_CLAIMS_{clean_col_name(v)}{sfx}"] = int((sel["TREATMENT_CATEGORY"] == v).sum())
        for v in PIVOT_VALUES["brand"]:
            expected[f"TOTAL_RX_CLAIMS_{clean_col_name(v)}{sfx}"] = int((sel["_BRAND"] & (sel["Category"] == v)).sum())
        for v in PIVOT_VALUES["rej_category"]:
            expected[f"TOTAL_REJECTED_RX_CLAIMS_{clean_col_name(v)}{sfx}"] = int((sel["_REJ"] & (sel["TREATMENT_CATEGORY"] == v)).sum())
        for fname, raw_n in expected.items():
            n_checks += 1
            if _v(feat, fname) != raw_n:
                n_bad += 1
                bad.append({"COHORT_ID": cid, "feature": fname, "raw": raw_n, "feature_value": _v(feat, fname)})
        row[f"{w}M_raw"]     = len(sel)
        row[f"{w}M_feature"] = _v(feat, f"TOTAL_RX_CLAIMS{sfx}")
    run_raw = int((_raw[SVC_COL] <= pd.Timestamp(cb["cutoff"])).sum())
    n_checks += 1
    if _v(feat, "RUNNING_TOTAL_RX_CLAIMS") != run_raw:
        n_bad += 1
        bad.append({"COHORT_ID": cid, "feature": "RUNNING_TOTAL_RX_CLAIMS", "raw": run_raw,
                    "feature_value": _v(feat, "RUNNING_TOTAL_RX_CLAIMS")})
    row.update({"RUNNING_raw": run_raw, "RUNNING_feature": _v(feat, "RUNNING_TOTAL_RX_CLAIMS"),
                "checks": n_checks, "mismatches": n_bad})
    rows.append(row)

trace = pd.DataFrame(rows)
print(f"Trace NPI: {TRACE_NPI} | cohorts: {len(trace)} | checks: {trace['checks'].sum():,} | "
      f"mismatches: {trace['mismatches'].sum()}")
display(trace)
if bad:
    display(pd.DataFrame(bad).head(50))

### 4. Validation

1. **Offset = 0 parity (every cohort)**: rebuild every month at offset 0 for the test NPIs with `build_cohort`, and compare every column shared with Wenjie's `claim_rx_monthly` for that month. Window bounds snap to calendar months at offset 0, so values should match exactly. Original nulls are treated as 0. Windowed/RUNNING_ columns that exist only in the original (values absent for the test NPIs) must be 0.
2. **HCP trace (every cohort)**: for the busiest test NPI, recount raw Rx claims per cohort and window in pandas (independent of Spark) and compare totals, treatment category, brand and rejection counts and the running total with the written features.

In [0]:
# ---- Validation 1: offset = 0 parity vs original claim_rx_monthly, every cohort ----
import builtins   # round/sum are shadowed by a pyspark.sql.functions star import
import numpy as np
import pandas as pd

_npis = get_test_npis()
if _npis is None:
    print("Production mode — parity check skipped (runs on the test NPIs only)")
else:
    _orig_all = (
        spark.read.format("delta").load(_NOTEBOOK_OUTPUT + "claim_rx_monthly")
        .filter(F.col(NPI_COL).isin(_npis))
        .withColumn("_M", F.date_format("SVC_DT_M", "yyyy-MM"))
    )
    _orig_months = {r[0] for r in _orig_all.select("_M").distinct().collect()}
    # Same cohort range, with the complete-data rule evaluated at offset 0
    PARITY_MONTHS = [m for m in get_cohort_months(max_data_date=DATA_MAX_DT, offset_days=0) if m in _orig_months]
    pd_orig_all = _orig_all.filter(F.col("_M").isin(PARITY_MONTHS)).toPandas()

    _skip = {"SVC_DT_M", "PULL_DT", "_M", NPI_COL, "PROVIDER_ID",
             "COHORT_ID", "COHORT_MONTH", "OFFSET_DAYS", "CUTOFF_DATE"}
    def _is_window_col(c):
        return "_IN_LAST_" in c or "_IN_PRIOR_" in c or c.startswith("RUNNING_")
    def _num(df):
        # Column-wise pd.to_numeric over ~3,700 columns took ~85 s per month; convert only the few
        # object columns that way and cast the rest in one go (same values).
        obj = list(df.columns[(df.dtypes == object).values])   # vectorised (df[c] per column is slow here)
        if obj:
            df = df.copy()
            df[obj] = df[obj].apply(pd.to_numeric, errors="coerce")
        return df.astype(float).fillna(0)

    # Same inputs as the original: full Rx history (no DATA_START_DATE) + the 3.0 patient-treatment
    # table (calendar months). This checks the feature LOGIC; the 2023 start is a config choice.
    _pat30 = (spark.read.format("delta").load(_QUERY_OUTPUT + get_patient_table_30("new_and_total_treatments"))
              .withColumn("SVC_DT_M", F.to_date("SVC_DT_M")))
    FAST_SOURCES["orig"] = lambda: (df_rx_all, _pat30)

    # Known quirk of the original: its RUNNING_ ..._SGLT2_WO_INVOKANA_300MG columns can differ from its own
    # RUNNING_ SGLT2 minus RUNNING_ INVOKANA_300MG (e.g. NPI 1386651560: 58 rejected SGLT2, 0 rejected
    # Invokana 300 -> original says 57). 3.1 keeps the consistent value; these cells are reported apart.
    def _is_quirk(c):
        return c.startswith("RUNNING_") and "SGLT2_WO_INVOKANA_300MG" in c

    # All parity months in ONE query (same engine as cell 31), then compared month by month in pandas
    _t_q = time.time()
    _cbs0 = {m: get_cohort_bounds(m, offset_days=0) for m in PARITY_MONTHS}
    pd_new_all = build_all_cohorts_fast(list(_cbs0.values()), source="orig").toPandas()
    print(f"{len(PARITY_MONTHS)} months built in one query: {time.time() - _t_q:,.0f}s")

    summary, mismatches = [], []
    for m in PARITY_MONTHS:
        _t0 = time.time()
        cb0 = _cbs0[m]
        pd_new  = pd_new_all[pd_new_all["COHORT_ID"] == cb0["cohort_id"]].set_index(NPI_COL)
        pd_orig = pd_orig_all[pd_orig_all["_M"] == m].set_index(NPI_COL)

        common    = sorted(set(pd_new.index) & set(pd_orig.index))
        only_orig = sorted(set(pd_orig.index) - set(pd_new.index))
        only_new  = sorted(set(pd_new.index) - set(pd_orig.index))
        cols      = sorted((set(pd_new.columns) & set(pd_orig.columns)) - _skip)

        o = _num(pd_orig.loc[common, cols])
        n = _num(pd_new.loc[common, cols])
        bad_all = ~np.isclose(o.values, n.values, rtol=1e-9, atol=1e-9)
        _q = np.array([_is_quirk(c) for c in cols], dtype=bool)
        bad, bad_quirk = bad_all & ~_q, bad_all & _q
        for i, j in list(zip(*np.where(bad_all)))[:50]:
            mismatches.append({"cohort": cb0["cohort_id"], "NPI": common[i], "column": cols[j],
                               "original": o.values[i, j], "new": n.values[i, j],
                               "known_quirk": bool(_q[j])})

        # Window/RUNNING_ columns only in the original (values absent for the test NPIs) must be 0
        _orig_only = sorted(c for c in set(pd_orig.columns) - set(pd_new.columns) - _skip if _is_window_col(c))
        _orig_only_nonzero = int((_num(pd_orig.loc[common, _orig_only]) != 0).values.sum()) if _orig_only else 0
        # NPIs only in the original for this month (no Rx history yet) must be all zeros
        _only_orig_nonzero = (int((_num(pd_orig.loc[only_orig, [c for c in pd_orig.columns if _is_window_col(c)]]) != 0)
                                  .any(axis=1).sum()) if only_orig else 0)

        summary.append({"cohort_offset0": cb0["cohort_id"], "cutoff": f"{cb0['cutoff']:%Y-%m-%d}",
                        "npis_common": len(common), "npis_only_original": len(only_orig),
                        "only_original_nonzero": _only_orig_nonzero, "npis_only_new": len(only_new),
                        "columns_compared": len(cols), "cells_compared": int(bad.size),
                        "mismatching_cells": int(bad.sum()), "known_quirk_cells": int(bad_quirk.sum()),
                        "orig_only_window_cols": len(_orig_only), "orig_only_nonzero_cells": _orig_only_nonzero,
                        "seconds": builtins.round(time.time() - _t0)})
        print(f"{cb0['cohort_id']}: {int(bad.sum())} mismatching cells, {int(bad_quirk.sum())} known-quirk "
              f"({time.time() - _t0:,.0f}s)")

    df_parity = pd.DataFrame(summary)
    display(df_parity)
    _ok = (df_parity[["mismatching_cells", "only_original_nonzero", "npis_only_new",
                      "orig_only_nonzero_cells"]].values.sum() == 0)
    print(f"Cohorts compared: {len(df_parity)} | cells compared: {df_parity['cells_compared'].sum():,} | "
          f"mismatching: {df_parity['mismatching_cells'].sum()} | known-quirk: {df_parity['known_quirk_cells'].sum()} | "
          f"{'PASS' if _ok else 'CHECK MISMATCHES'}")
    if mismatches:
        display(pd.DataFrame(mismatches))

## Part 4 -- Status and Flags for Wenjie

### Resolved in This Notebook

* **`RUNNING_` features** are now included. A dedicated cell counts all claims from the start of `Rx_01_new` history through `CUTOFF_DATE` per NPI, with null NPIs excluded. It covers every feature group the old notebook had (treatment category, brand, rejection, duration, brand x refill, refill alone, payer, patient attributes, patient treatment buckets) plus the `SGLT2_WO_INVOKANA_300MG` derived running columns.
* **AVG extra-month semantics** are controlled by `AVG_EXTRA_MONTH` (default `True` = old behavior). A two-stage cell aggregates per NPI per calendar month, zero-fills missing months, then divides by the fixed denominator (N+1 when True, N when False). This replicates the old `step3_lag_avg` behavior exactly.
* **Treatment x refill columns** removed from the refill group. Old notebook had brand x refill + refill alone only; the extra treatment x refill columns were new and are now gone.
* **Window boundary snap at offset 0**: `_build_bounds` snaps exclusive start to `first_of(cohort_month - (N-1 months)) - 1 day`, so windows align exactly with calendar months. No 1-2 day rounding differences.
* **`.cache()` removed**: Not supported on serverless.
* **`import builtins`** guards against `00_connection` shadowing Python `max`/`min`/`sum`.

### Still Open -- Discuss with Wenjie

1. ~~**`pat_new_and_total_treatments` month alignment**~~ **RESOLVED**: Added `assign_cohort_month()` helper (mirrors `00_cohort_config`) and replaced all `date_trunc('MM', ...)` with it in cells 18, 19, and 20. At offset>0, claims now map to the same cohort month as the patient journey pipeline, so the join is correct.

2. **NPI to BH_ID mapping**: The original notebook joins this at the end (cells 73-74). Not yet added to this notebook. Add when the mapping table path is confirmed.

3. **`99_Combine_Physician_Data` wiring**: The combine notebook joins on `SVC_DT_M`. This notebook outputs `COHORT_ID` instead. Decide how to bridge: (a) derive `SVC_DT_M` from cohort config, or (b) switch the join key.

4. **`00_step23_function`** helpers (`step2_fill_missing`, `step3_lag`, `step3_lag_avg`) are **not used** here. Other notebooks (02-06) still depend on them.

### Nice to Have

5. **Multi-cohort generation**: One cohort per run. For training, extend with a parameterized Lakeflow Job.
6. **Patient journey provenance**: Locate where the claim-eligible patient file is created upstream.
7. **Columns from old LAD V6**: Cross-check column lists when validating in case of missing switch/add-on columns.